# Siamese CNN (from scratch): Unified Multi-Dataset & Function-Callable Notebook

Face analysis project supporting **UTKFace**, **FairFace** (`HuggingFaceM4/FairFace` via Hugging Face `datasets` or `.zip` upload, with `unified_5` or `native` 7-class race taxonomy), **Combined (UTKFace + FairFace)**, and **Cross-Dataset Evaluation**.

You can use this notebook in **two ways**:
1. **Top-to-bottom interactive mode:** Set switches/sources in **Step 0** and **Step 2**, then run every cell sequentially; each step finishes with automated `PASS` / `FAIL` checks.
2. **Single-function programmatic mode (`run_experiment`):** Call `run_experiment(...)` from anywhere (or the final cell) with dataset selection, FairFace source/subset (`"0.25"` or `"1.25"`), `epochs`, `steps_per_epoch`, learning rate, ablation flags, etc., and receive all classification reports, verification metrics, fairness tables, training history, and saved model checkpoints in one structured dictionary.

| Step | What | Validation |
|---|---|---|
| 0 | Setup, unified `Config`, `CheckTracker`, device | seeding, device, output/cache dirs, config overrides |
| 1 | Shared label schema & `DatasetSpec` (`unified_5` & `native`) | age-bin edges, 5-class & 7-class label maps |
| 2 | Dataset paths & source resolution (UTKFace, FairFace HF/zip, Combined) | source resolution, path & library checks |
| 3 | Dataset loaders & canonical schema normalization | filename / HF `datasets` / zip parsing, label ranges |
| 4 | Deduplication, near-duplicate audit, stratified / official / cross-dataset split (`prepare_datasets`) | no leakage, Hamming distance test, class coverage |
| 5 | Disk-backed GPU image cache (`build_cache`) | shapes, blank images, reload parity |
| 6 | GPU augmentation (mild & strong, per-sample randomness) | per-sample randomness, identity, strong variance |
| 7 | Pair sampler (attribute pairs, dynamic `n_race` x `n_age`) | positive rate, pair rules, difficulty stratification |
| 8 | Model & losses (dual projection heads + dynamic attribute heads) | size, shapes, NT-Xent, gradient isolation, overfit test |
| 9 | Training loop & evaluation helpers | mini fit, resume, bootstrap CIs, balanced fairness, probes |
| 10 | `run_siamese_cnn(data, config)` | core model training & evaluation engine |
| 11 | Full model run, validation & plots | structure, ranges, bootstrap CIs, config B fallback, plots |
| 12 | A / B / C ablation over seeds (`run_ablation`) | aggregation, resumable CSV, mean +- std table & bars |
| 13 | Inference driver (`run_siamese_pipeline`) | agrees with batch evaluation, self-comparison, input types |
| 14 | Unified end-to-end notebook entrypoint (`run_experiment`) | full pipeline execution across datasets & configs |

## Step 0 - Setup

In [ ]:
import os, sys, json, time, math, random, re, copy, hashlib, shutil, warnings
import dataclasses
from dataclasses import dataclass, asdict, field
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
                             balanced_accuracy_score, classification_report,
                             cohen_kappa_score, confusion_matrix, log_loss,
                             matthews_corrcoef, precision_recall_curve,
                             precision_recall_fscore_support, roc_auc_score,
                             roc_curve, top_k_accuracy_score)
from sklearn.model_selection import train_test_split

# ---------------- global interactive defaults ----------------
QUICK_RUN = True     # True: ~3000 images, 2 epochs. False: full run.
USE_FAIRFACE = True   # Legacy interactive switch; prefer Config(dataset="fairface" | "combined")
STOP_ON_FAIL = True    # Stop immediately on the first failed validation check.
RUN_ABLATION = False   # Run A/B/C ablation over seeds in Step 12.

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
class CheckTracker:
    """Run-scoped validation check recorder (keeps interactive global CHECKS in sync)."""
    def __init__(self, stop_on_fail: bool = True, sync_global: bool = True):
        self.stop_on_fail = stop_on_fail
        self.sync_global = sync_global
        self.records: List[Dict[str, Any]] = []

    def check(self, name: str, cond: Any, detail: Any = "") -> bool:
        ok = bool(cond)
        rec = {"check": name, "passed": ok, "detail": str(detail)}
        self.records.append(rec)
        if self.sync_global:
            CHECKS.append(rec)
        print(("PASS  " if ok else "FAIL  ") + name + (f"   [{detail}]" if str(detail) != "" else ""))
        if not ok and self.stop_on_fail:
            raise AssertionError(f"Check failed: {name} {detail}")
        return ok

    def summary(self) -> Dict[str, Any]:
        total = len(self.records)
        passed = sum(1 for r in self.records if r["passed"])
        failed = [r for r in self.records if not r["passed"]]
        return {
            "passed": passed,
            "failed": len(failed),
            "total": total,
            "all_passed": len(failed) == 0,
            "failures": failed,
        }

CHECKS: List[Dict[str, Any]] = []
DEFAULT_TRACKER = CheckTracker(stop_on_fail=STOP_ON_FAIL, sync_global=True)

def check(name: str, cond: Any, detail: Any = "", tracker: Optional[CheckTracker] = None) -> bool:
    """Record and print a validation result; stop on failure if stop_on_fail is active."""
    t = tracker if tracker is not None else DEFAULT_TRACKER
    t.stop_on_fail = STOP_ON_FAIL if tracker is None else t.stop_on_fail
    return t.check(name, cond, detail)

In [ ]:
UNIFIED_RACE_NAMES = ("White", "Black", "Asian", "Indian", "Others")
FAIRFACE_NATIVE_RACE_NAMES = (
    "White", "Black", "East Asian", "Southeast Asian",
    "Indian", "Middle Eastern", "Latino_Hispanic"
)

@dataclass
class Config:
    # Dataset & source configuration
    dataset: str = "fairface"                # "utkface", "fairface", or "combined"
    utk_dir: str = "/kaggle/input/datasets/jangedoo/utkface-new/UTKFace"
    fairface_source: str = "auto"           # "auto" (checks .zip first, else HF load_dataset), "hf", or path to .zip
    fairface_config: str = "0.25"           # "0.25" or "1.25" (matches load_dataset("HuggingFaceM4/FairFace", ...))
    eval_dataset: Optional[str] = None      # Optional cross-dataset evaluation target ("utkface" or "fairface")
    race_schema: str = "native"          # "unified_5" (5 shared classes) or "native" (FairFace 7 classes)
    split_mode: str = "stratified"          # "stratified" (70/15/15) or "official" (uses FairFace val as test)
    drop_near_dups: bool = False
    near_dup_max_dist: int = 4

    # Architecture & label dimensions (auto-synced to DatasetSpec during dataset prep)
    img_size: int = 64
    feat_dim: int = 256                     # backbone features (feed the attribute heads)
    emb_dim: int = 128                      # L2-normalised embedding dimension for each projection head
    n_race: int = 5
    n_age: int = 9
    race_names: Tuple[str, ...] = UNIFIED_RACE_NAMES

    # Training hyperparameters & loop controls
    epochs: int = 15
    steps_per_epoch: Optional[int] = 300    # int = fixed steps/epoch; None = auto (full pass over train)
    batch_pairs: int = 64                   # 64 attribute pairs = 128 images per step
    batch_inst: int = 64                    # 64 instance images per step (2 strong views each = 128 images)
    lr: float = 2e-3
    weight_decay: float = 1e-4
    margin: float = 1.0
    w_pair: float = 1.0                     # attribute-pair contrastive loss weight
    w_inst: float = 1.0                     # instance-pair (NT-Xent) loss weight
    attr_weight: float = 1.0                # attribute heads cross-entropy loss weight
    temperature: float = 0.2                # NT-Xent temperature
    inst_strength: float = 2.5              # augmentation strength for instance views
    group_power: float = 0.5                # group sampling prob ~ size ** group_power

    # Evaluation, bootstrap & runtime flags
    n_eval_pairs: int = 5000                # fixed validation / test attribute pairs
    n_inst_eval: int = 2000                 # images used for instance verification
    eval_seed: int = 1000                   # fixed eval pairs, independent of training seed
    n_boot: int = 500                       # image-level bootstrap replicates
    seed: int = 42
    amp: bool = True
    quick_run: bool = False
    quick_total: int = 3000                 # images sampled when quick_run=True
    resume: bool = False
    early_stop_patience: int = 4            # stop after N epochs without val_loss improvement (0 = off)
    early_stop_min_delta: float = 1e-4
    run_ablation: bool = False
    ablation_seeds: Tuple[int, ...] = (0, 1, 2)
    show_plots: bool = True
    run_validation: bool = True
    stop_on_fail: bool = True
    out_dir: str = str(WORK_ROOT / "siamese_cnn")
    cache_dir: Optional[str] = str(WORK_ROOT / "cache")
    name: str = "C_full"

def sync_config_schema(c: Config) -> Config:
    """Ensure race_names and n_race stay consistent with dataset and race_schema."""
    if c.race_schema == "native" and c.dataset.lower() == "fairface":
        c.race_names = FAIRFACE_NATIVE_RACE_NAMES
    elif c.race_schema == "unified_5":
        c.race_names = UNIFIED_RACE_NAMES
    else:
        raise ValueError(f"Unsupported combination: dataset={c.dataset!r}, race_schema={c.race_schema!r}")
    c.race_names = tuple(c.race_names)
    c.n_race = len(c.race_names)
    return c

def make_config(base=None, **overrides):
    # 1. Start from a fresh Config() (or a copy of `base` if explicitly passed)
    c = copy.deepcopy(base) if base is not None else Config()

    # 2. Sync any top-level global variables if they are defined in the notebook
    global_map = {
        "DATASET": "dataset",
        "RACE_SCHEMA": "race_schema",
        "QUICK_RUN": "quick_run",
        "SPLIT_MODE": "split_mode",
        "EVAL_DATASET": "eval_dataset",
    }
    if base is None:
        for g_name, field_name in global_map.items():
            if g_name in globals() and hasattr(c, field_name):
                setattr(c, field_name, globals()[g_name])

    # 3. Apply any explicit keyword overrides (e.g., make_config(dataset="utkface"))
    for k, v in overrides.items():
        setattr(c, k, v)

    # 4. Auto-fix race_schema when dataset changes so sync_config_schema never crashes
    if "race_schema" not in overrides:
        if c.dataset in ("utkface", "combined") and c.race_schema == "native":
            c.race_schema = "unified_5"
        elif "dataset" in overrides and c.dataset == "fairface":
            c.race_schema = "native"

    return sync_config_schema(c)

def _json_default(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, Path): return str(o)
    return str(o)

cfg = make_config()
print("dataset =", cfg.dataset, "| race_schema =", cfg.race_schema, "| quick_run =", cfg.quick_run)
print(cfg)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(s: int):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

cfg.amp = cfg.amp and DEVICE.type == "cuda"
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "NO GPU")

**Validation - Step 0**

In [ ]:
set_seed(cfg.seed); a = torch.rand(4)
set_seed(cfg.seed); b = torch.rand(4)
check("seeding is reproducible", torch.equal(a, b))
check("device selected", True, DEVICE)
if DEVICE.type == "cpu":
    print("WARNING: no GPU. Enable an accelerator in Kaggle (Session options) for full runs.")
for d in filter(None, (cfg.out_dir, cfg.cache_dir)):
    Path(d).mkdir(parents=True, exist_ok=True)
    (Path(d) / ".w").write_text("x"); (Path(d) / ".w").unlink()
check("output and cache dirs writable", True, f"{cfg.out_dir} | {cfg.cache_dir}")
check("variants via dataclasses.replace keep base values",
      dataclasses.replace(cfg, w_inst=0.0).batch_pairs == cfg.batch_pairs)
ff_cfg = make_config(dataset="fairface", race_schema="native")
check("make_config auto-syncs FairFace native 7-class race schema",
      ff_cfg.n_race == 7 and len(ff_cfg.race_names) == 7)

## Step 1 - Shared label schema & `DatasetSpec`

All datasets normalize into a canonical schema (`path, age, gender, race, age_bin, dataset, race7`) paired with a `DatasetSpec` metadata object:
* **9 age bins** (`0-2` through `70+`).
* **Race taxonomy:** either **`unified_5`** (`White, Black, Asian, Indian, Others` — maps FairFace's 7 classes into UTKFace's 5 so datasets can be combined or cross-evaluated) or **`native`** (preserves FairFace's 7 classes or custom dataset classes).
* **Gender:** `0 = male`, `1 = female`.

In [ ]:
AGE_EDGES = [3, 10, 20, 30, 40, 50, 60, 70]
AGE_BIN_NAMES = ["0-2", "3-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"]
AGE_BIN_MIDPOINTS = np.array([1, 6, 15, 25, 35, 45, 55, 65, 75], dtype=float)
RACE_NAMES = list(cfg.race_names)          # Dynamic default; overridden per DatasetSpec
GENDER_NAMES = ["male", "female"]

# FairFace 7 classes -> harmonised 5 (East/Southeast Asian -> Asian; Middle Eastern/Latino -> Others)
FAIRFACE_RACE_MAP_UNIFIED = {
    "White": 0, "Black": 1, "East Asian": 2, "Southeast Asian": 2,
    "Indian": 3, "Middle Eastern": 4, "Latino_Hispanic": 4,
}
FAIRFACE_RACE_MAP_NATIVE = {name: i for i, name in enumerate(FAIRFACE_NATIVE_RACE_NAMES)}
FAIRFACE_RACE_MAP = FAIRFACE_RACE_MAP_UNIFIED

FAIRFACE_AGE_MAP = {
    "0-2": 0, "3-9": 1, "10-19": 2, "20-29": 3, "30-39": 4,
    "40-49": 5, "50-59": 6, "60-69": 7, "more than 70": 8, "70+": 8,
}
FAIRFACE_GENDER_MAP = {"Male": 0, "Female": 1, "male": 0, "female": 1, "0": 0, "1": 1}

@dataclass
class DatasetSpec:
    name: str
    race_schema: str
    race_names: Tuple[str, ...]
    gender_names: Tuple[str, ...] = ("male", "female")
    age_bin_names: Tuple[str, ...] = tuple(AGE_BIN_NAMES)
    has_continuous_age: bool = True

    @property
    def num_races(self) -> int:
        return len(self.race_names)

    @property
    def num_ages(self) -> int:
        return len(self.age_bin_names)

def age_to_bin(ages):
    return np.digitize(np.asarray(ages), AGE_EDGES)

**Validation - Step 1**

In [ ]:
pairs = [(0, 0), (2, 0), (3, 1), (9, 1), (10, 2), (29, 3), (30, 4), (69, 7), (70, 8), (116, 8)]
bad = [(a, age_to_bin(a), b) for a, b in pairs if int(age_to_bin(a)) != b]
check("age_to_bin edge cases", not bad, bad)
check("bin names / midpoints / edges consistent",
      len(AGE_BIN_NAMES) == len(AGE_BIN_MIDPOINTS) == len(AGE_EDGES) + 1 == cfg.n_age)
check("race names match config", len(RACE_NAMES) == cfg.n_race)
check("FairFace unified race map covers 7 classes -> 5",
      len(FAIRFACE_RACE_MAP_UNIFIED) == 7 and set(FAIRFACE_RACE_MAP_UNIFIED.values()) == set(range(5)))
check("FairFace native race map covers 7 classes -> 7",
      len(FAIRFACE_RACE_MAP_NATIVE) == 7 and set(FAIRFACE_RACE_MAP_NATIVE.values()) == set(range(7)))

## Step 2 - Dataset paths & source resolution

Configure `UTK_DIR` and `FAIRFACE_SOURCE` (`"auto"`, `"hf"`, or a `.zip` path) below for interactive runs, or pass `dataset=...`, `utk_dir=...`, `fairface_source=...`, and `fairface_config=...` directly to `run_experiment(...)`.

In [ ]:
import zipfile

IMG_EXT = {".jpg", ".jpeg", ".png"}
UTK_NAME = re.compile(r"^(\d{1,3})_([01])_([0-4])_")   # age_gender_race_date

# ===================== SET YOUR INTERACTIVE SOURCES HERE =====================
UTK_DIR = cfg.utk_dir
FAIRFACE_SOURCE = cfg.fairface_source   # "auto", "hf", or "/path/to/fairface.zip"
FAIRFACE_CONFIG = cfg.fairface_config   # "0.25" or "1.25"
# =============================================================================

cfg.utk_dir = UTK_DIR
cfg.fairface_source = FAIRFACE_SOURCE
cfg.fairface_config = FAIRFACE_CONFIG

def show_input_tree(root, max_children=8):
    root = Path(root)
    if not root.exists():
        print(f"{root} does not exist (add a dataset with '+ Add Input', or pass paths to run_experiment)"); return
    kids = sorted(root.iterdir())
    if not kids:
        print("(/kaggle/input is empty: no dataset has been added to this notebook)")
    for d in kids:
        print(d.name)
        if d.is_dir():
            sub = sorted(k.name for k in d.iterdir())
            print("   ", sub[:max_children], "..." if len(sub) > max_children else "")

def resolve_fairface_source(source: str) -> Tuple[str, str]:
    """Return (kind, target) where kind in {'zip', 'dir', 'hf', 'missing'}."""
    src = (source or "auto").strip()
    if src.lower() in ("hf", "huggingface", "huggingfacem4/fairface"):
        return "hf", "HuggingFaceM4/FairFace"
    if src.lower() != "auto":
        p = Path(src)
        if p.is_file() and p.suffix.lower() == ".zip":
            return "zip", str(p)
        if p.is_dir():
            return "dir", str(p)
        return "missing", src

    # Auto-mode: check if a FairFace .zip archive was uploaded first; otherwise use HF datasets
    for root in (r for r in (INPUT_ROOT, WORK_ROOT, Path(".")) if r.exists()):
        for entry in sorted(root.rglob("*.zip")):
            if len(entry.relative_to(root).parts) <= 3 and "fairface" in entry.name.lower():
                return "zip", str(entry)
    return "hf", "HuggingFaceM4/FairFace"

def inspect_dataset_paths(c: Config, verbose: bool = True) -> Dict[str, Any]:
    ds = c.dataset.lower()
    ev = c.eval_dataset.lower() if c.eval_dataset else None
    ff_kind, ff_target = resolve_fairface_source(c.fairface_source)
    info = {"dataset": ds, "utk_matching": 0, "ff_kind": ff_kind, "ff_target": ff_target, "hf_ready": False}

    if "utkface" in (ds, ev) or ds == "combined":
        if os.path.isdir(c.utk_dir):
            info["utk_matching"] = sum(1 for f in os.listdir(c.utk_dir) if UTK_NAME.match(f))
            if verbose:
                print("UTKFace folder:", c.utk_dir, "| images matching age_gender_race_*:", info["utk_matching"])
                if info["utk_matching"] == 0:
                    print("Folder exists but no file matches. First files found:", sorted(os.listdir(c.utk_dir))[:5])
        elif verbose:
            print(f"UTK_DIR does not exist: {c.utk_dir}\nWhat is under {INPUT_ROOT}:")
            show_input_tree(INPUT_ROOT)

    if "fairface" in (ds, ev) or ds == "combined":
        try:
            import datasets  # noqa: F401
            info["hf_ready"] = True
        except ImportError:
            info["hf_ready"] = False
        if verbose:
            if ff_kind == "hf":
                print(f'FairFace source -> load_dataset("HuggingFaceM4/FairFace", "{c.fairface_config}") | datasets installed: {info["hf_ready"]}')
            else:
                print(f"FairFace source -> [{ff_kind}] {ff_target}")
    return info

path_info = inspect_dataset_paths(cfg, verbose=True)
n_matching = path_info["utk_matching"]
utk_counts = Counter({cfg.utk_dir: n_matching})

**Validation - Step 2**

In [ ]:
ds_active = cfg.dataset.lower()
if ds_active in ("utkface", "combined"):
    check("UTKFace folder exists", os.path.isdir(cfg.utk_dir), cfg.utk_dir)
    check("UTKFace has a plausible number of images", n_matching >= (200 if cfg.quick_run else 20000),
          f"{n_matching} images (full UTKFace is about 23,700)")
else:
    print(f"SKIP  UTKFace checks (dataset = {cfg.dataset})")

if ds_active in ("fairface", "combined"):
    check("FairFace source resolved", path_info["ff_kind"] in ("zip", "dir", "hf"),
          f"{path_info['ff_kind']}: {path_info['ff_target']}")
    if path_info["ff_kind"] == "hf":
        check("Hugging Face 'datasets' library installed", path_info["hf_ready"])
        check("Valid FairFace config ('0.25' or '1.25')", cfg.fairface_config in ("0.25", "1.25"), cfg.fairface_config)
else:
    print(f"SKIP  FairFace checks (dataset = {cfg.dataset})")

## Step 3 - Loaders & dataset adapters

Every adapter returns `(df, spec, meta)` where `df` follows the canonical schema (`path, age, gender, race, age_bin, dataset, race7`, plus optional `split` when `split_mode="official"`) and `spec` is a `DatasetSpec`.

In [ ]:
HF_FAIRFACE_RACE_NAMES = ("East Asian", "Indian", "Black", "White", "Middle Eastern", "Latino_Hispanic", "Southeast Asian")
HF_FAIRFACE_AGE_NAMES = ("0-2", "3-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "more than 70")
HF_FAIRFACE_GENDER_NAMES = ("Male", "Female")

def load_utkface(utk_dir: str) -> Tuple[pd.DataFrame, List[str]]:
    rows, dropped = [], []
    for f in sorted(os.listdir(utk_dir)):
        if os.path.splitext(f.lower())[1] not in IMG_EXT:
            continue
        m = UTK_NAME.match(f)
        if not m:
            dropped.append(f); continue
        age, gender, race = map(int, m.groups())
        rows.append((os.path.join(utk_dir, f), float(age), gender, race))
    df = pd.DataFrame(rows, columns=["path", "age", "gender", "race"])
    df["age_bin"] = age_to_bin(df["age"]) if len(df) else np.array([], dtype=int)
    df["dataset"], df["race7"] = "utkface", None
    return df, dropped

def _load_fairface_hf_dataset(
    subset: str = "0.25",
    cache_dir: Optional[str] = None,
    race_map: Dict[str, int] = FAIRFACE_RACE_MAP_UNIFIED,
    hf_ds: Optional[Any] = None,
    n_threads: int = 8,
) -> Tuple[pd.DataFrame, int]:
    from datasets import Image as HFImage, load_dataset

    ds_dict = hf_ds if hf_ds is not None else load_dataset("HuggingFaceM4/FairFace", subset)
    img_dir = Path(cache_dir or (WORK_ROOT / "cache")) / f"fairface_hf_{subset.replace('.', '_')}"
    img_dir.mkdir(parents=True, exist_ok=True)

    frames = []
    for hf_split, mapped_split in (("train", "train"), ("validation", "test")):
        if hf_split not in ds_dict:
            continue
        split_ds = ds_dict[hf_split]
        r_names = split_ds.features["race"].names
        a_names = split_ds.features["age"].names
        g_names = split_ds.features["gender"].names

        race_strs = [r_names[i] for i in split_ds["race"]]
        age_strs = [a_names[i] for i in split_ds["age"]]
        gender_strs = [g_names[i] for i in split_ds["gender"]]

        raw_imgs = split_ds.cast_column("image", HFImage(decode=False))["image"]
        paths = [str(img_dir / f"{hf_split}_{idx:06d}.jpg") for idx in range(len(split_ds))]

        def _write_one(item):
            p, img_obj = item
            if not os.path.exists(p):
                if img_obj.get("bytes") is not None:
                    with open(p, "wb") as f:
                        f.write(img_obj["bytes"])
                elif img_obj.get("path") and os.path.exists(img_obj["path"]):
                    shutil.copyfile(img_obj["path"], p)

        with ThreadPoolExecutor(n_threads) as ex:
            list(ex.map(_write_one, zip(paths, raw_imgs)))

        frames.append(pd.DataFrame({
            "path": paths,
            "age": np.nan,
            "gender": [FAIRFACE_GENDER_MAP[g] for g in gender_strs],
            "race": [race_map[r] for r in race_strs],
            "age_bin": [FAIRFACE_AGE_MAP[a] for a in age_strs],
            "dataset": "fairface",
            "race7": race_strs,
            "split": mapped_split,
        }))

    out = pd.concat(frames, ignore_index=True)
    exists_mask = out["path"].map(os.path.exists)
    return out[exists_mask].reset_index(drop=True), int((~exists_mask).sum())

def _extract_zip_once(zip_path: str, cache_dir: Optional[str]) -> str:
    zp = Path(zip_path)
    out_root = Path(cache_dir or (WORK_ROOT / "cache")) / f"fairface_zip_{zp.stem}"
    marker = out_root / ".extracted_ok"
    if not marker.exists():
        out_root.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {zp.name} -> {out_root} ...")
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(out_root)
        marker.touch()
    return str(out_root)

def _load_fairface_from_dir(
    root_dir: str, subset: str, cache_dir: Optional[str], race_map: Dict[str, int], n_threads: int = 8
) -> Tuple[pd.DataFrame, int]:
    root = Path(root_dir)
    # 1. Zipped HuggingFace Parquet shards
    pq_files = sorted(root.rglob("*.parquet"))
    if pq_files:
        subset_pqs = [p for p in pq_files if subset in p.parts]
        pq_files = subset_pqs or pq_files
        img_dir = Path(cache_dir or (WORK_ROOT / "cache")) / f"fairface_pq_{subset.replace('.', '_')}"
        img_dir.mkdir(parents=True, exist_ok=True)
        frames = []
        for pq in pq_files:
            mapped_split = "test" if any(k in pq.stem.lower() for k in ("val", "test")) else "train"
            tbl = pd.read_parquet(pq)
            race_strs = [HF_FAIRFACE_RACE_NAMES[int(v)] if isinstance(v, (int, np.integer)) else str(v) for v in tbl["race"]]
            age_strs = [HF_FAIRFACE_AGE_NAMES[int(v)] if isinstance(v, (int, np.integer)) else str(v) for v in tbl["age"]]
            gender_strs = [HF_FAIRFACE_GENDER_NAMES[int(v)] if isinstance(v, (int, np.integer)) else str(v) for v in tbl["gender"]]
            paths = [str(img_dir / f"{pq.stem}_{idx:06d}.jpg") for idx in range(len(tbl))]
            def _write_bytes(item):
                p, obj = item
                if not os.path.exists(p):
                    raw = obj["bytes"] if isinstance(obj, dict) and obj.get("bytes") is not None else obj
                    if isinstance(raw, (bytes, bytearray)):
                        with open(p, "wb") as f:
                            f.write(raw)
            with ThreadPoolExecutor(n_threads) as ex:
                list(ex.map(_write_bytes, zip(paths, tbl["image"].tolist())))
            frames.append(pd.DataFrame({
                "path": paths, "age": np.nan,
                "gender": [FAIRFACE_GENDER_MAP[g] for g in gender_strs],
                "race": [race_map[r] for r in race_strs],
                "age_bin": [FAIRFACE_AGE_MAP[a] for a in age_strs],
                "dataset": "fairface", "race7": race_strs, "split": mapped_split,
            }))
        out = pd.concat(frames, ignore_index=True)
        exists_mask = out["path"].map(os.path.exists)
        return out[exists_mask].reset_index(drop=True), int((~exists_mask).sum())

    # 2. Standard FairFace zip (CSV label files + train/val image folders)
    img_lookup: Dict[str, str] = {}
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXT:
            img_lookup["/".join(p.parts[-2:])] = str(p)
            img_lookup.setdefault(p.name, str(p))

    frames = []
    for csv_path in sorted(root.rglob("*.csv")):
        name_l = csv_path.stem.lower()
        mapped_split = "train" if "train" in name_l else ("test" if ("val" in name_l or "test" in name_l) else None)
        if mapped_split is None:
            continue
        df = pd.read_csv(csv_path)
        if not {"file", "age", "gender", "race"}.issubset(df.columns):
            continue
        def _resolve_img(rel: str) -> str:
            rel_clean = str(rel).strip().lstrip("./")
            direct = csv_path.parent / rel_clean
            if direct.exists():
                return str(direct)
            return img_lookup.get("/".join(Path(rel_clean).parts[-2:]), img_lookup.get(Path(rel_clean).name, str(direct)))
        race_strs = df["race"].astype(str).str.strip().tolist()
        frames.append(pd.DataFrame({
            "path": [_resolve_img(f) for f in df["file"]],
            "age": np.nan,
            "gender": df["gender"].astype(str).str.strip().map(FAIRFACE_GENDER_MAP).astype(int),
            "race": [race_map[r] for r in race_strs],
            "age_bin": df["age"].astype(str).str.strip().map(FAIRFACE_AGE_MAP).astype(int),
            "dataset": "fairface", "race7": race_strs, "split": mapped_split,
        }))
    if not frames:
        raise FileNotFoundError(f"No valid FairFace CSV or Parquet files found in {root_dir}")
    out = pd.concat(frames, ignore_index=True)
    exists_mask = out["path"].map(os.path.exists)
    return out[exists_mask].reset_index(drop=True), int((~exists_mask).sum())

def load_fairface(
    source: str = "auto",
    subset: str = "0.25",
    cache_dir: Optional[str] = None,
    race_schema: str = "unified_5",
    hf_ds: Optional[Any] = None,
) -> Tuple[pd.DataFrame, int]:
    if race_schema == "native":
        race_map = FAIRFACE_RACE_MAP_NATIVE
    elif race_schema == "unified_5":
        race_map = FAIRFACE_RACE_MAP_UNIFIED
    else:
        raise ValueError(f"Unsupported race_schema for FairFace: {race_schema!r}")

    if hf_ds is not None:
        return _load_fairface_hf_dataset(subset=subset, cache_dir=cache_dir, race_map=race_map, hf_ds=hf_ds)

    kind, target = resolve_fairface_source(source)
    if kind == "zip":
        target = _extract_zip_once(target, cache_dir)
        kind = "dir"
    if kind == "dir":
        return _load_fairface_from_dir(target, subset=subset, cache_dir=cache_dir, race_map=race_map)
    if kind == "hf":
        return _load_fairface_hf_dataset(subset=subset, cache_dir=cache_dir, race_map=race_map)
    raise FileNotFoundError(f"FairFace source not found: {source!r}")

def _adapt_utkface(c: Config) -> Tuple[pd.DataFrame, DatasetSpec, Dict[str, Any]]:
    if c.race_schema != "unified_5":
        raise ValueError("UTKFace has 5 race classes; use race_schema='unified_5'.")
    df, dropped = load_utkface(c.utk_dir)
    spec = DatasetSpec(name="utkface", race_schema="unified_5", race_names=UNIFIED_RACE_NAMES, has_continuous_age=True)
    return df, spec, {"utk_dropped": dropped, "dropped_or_missing": len(dropped)}

def _adapt_fairface(c: Config) -> Tuple[pd.DataFrame, DatasetSpec, Dict[str, Any]]:
    preloaded = globals().get("ds") if c.fairface_source in ("auto", "hf") else None
    if preloaded is not None and not hasattr(preloaded, "keys"):
        preloaded = None
    df, n_missing = load_fairface(
        source=c.fairface_source, subset=c.fairface_config, cache_dir=c.cache_dir,
        race_schema=c.race_schema, hf_ds=preloaded
    )
    r_names = FAIRFACE_NATIVE_RACE_NAMES if c.race_schema == "native" else UNIFIED_RACE_NAMES
    spec = DatasetSpec(name="fairface", race_schema=c.race_schema, race_names=r_names, has_continuous_age=False)
    return df, spec, {"fair_missing": n_missing, "dropped_or_missing": n_missing}

def _adapt_combined(c: Config) -> Tuple[pd.DataFrame, DatasetSpec, Dict[str, Any]]:
    if c.race_schema != "unified_5":
        raise ValueError("Combined UTKFace + FairFace requires race_schema='unified_5'.")
    u_df, _, u_meta = _adapt_utkface(c)
    f_df, _, f_meta = _adapt_fairface(c)
    df = pd.concat([u_df, f_df], ignore_index=True)
    spec = DatasetSpec(name="combined", race_schema="unified_5", race_names=UNIFIED_RACE_NAMES, has_continuous_age=True)
    return df, spec, {**u_meta, **f_meta, "dropped_or_missing": u_meta["dropped_or_missing"] + f_meta["dropped_or_missing"]}

DATASET_ADAPTERS: Dict[str, Callable[[Config], Tuple[pd.DataFrame, DatasetSpec, Dict[str, Any]]]] = {
    "utkface": _adapt_utkface,
    "fairface": _adapt_fairface,
    "combined": _adapt_combined,
}

def load_dataset_by_name(name: str, c: Config) -> Tuple[pd.DataFrame, DatasetSpec, Dict[str, Any]]:
    key = name.lower()
    if key not in DATASET_ADAPTERS:
        raise ValueError(f"Unknown dataset {name!r}. Choose from {sorted(DATASET_ADAPTERS)}.")
    return DATASET_ADAPTERS[key](c)

raw_df, dataset_spec, load_meta = load_dataset_by_name(cfg.dataset, cfg)
cfg.race_names = dataset_spec.race_names
cfg.n_race = dataset_spec.num_races
RACE_NAMES = list(dataset_spec.race_names)
utk_df = raw_df if cfg.dataset.lower() == "utkface" else None
utk_dropped = load_meta.get("utk_dropped", [])
print(f"Loaded {dataset_spec.name} ({dataset_spec.race_schema}): {len(raw_df)} images | "
      f"{dataset_spec.num_races} race classes: {list(dataset_spec.race_names)}")

**Validation - Step 3**

In [ ]:
def validate_schema(df: pd.DataFrame, name: str, spec: DatasetSpec, tracker: Optional[CheckTracker] = None):
    check(f"{name}: required columns", {"path", "age", "gender", "race", "age_bin", "dataset", "race7"} <= set(df.columns), tracker=tracker)
    check(f"{name}: no missing labels", df[["path", "gender", "race", "age_bin"]].notna().all().all(), tracker=tracker)
    check(f"{name}: gender in {{0,1}}", set(df.gender.unique()) <= {0, 1}, tracker=tracker)
    check(f"{name}: race in 0..{spec.num_races - 1}", df.race.between(0, spec.num_races - 1).all(), tracker=tracker)
    check(f"{name}: age_bin in 0..{spec.num_ages - 1}", df.age_bin.between(0, spec.num_ages - 1).all(), tracker=tracker)
    check(f"{name}: paths unique and exist",
          df.path.is_unique and all(os.path.exists(p) for p in df.path.sample(min(50, len(df)), random_state=0)),
          tracker=tracker)

validate_schema(raw_df, dataset_spec.name, dataset_spec)
if dataset_spec.name == "utkface":
    check("UTKFace: ages in 1..116", raw_df.age.between(1, 116).all(), f"min {raw_df.age.min()} max {raw_df.age.max()}")
    check("UTKFace: malformed share under 1%", len(utk_dropped) <= 0.01 * (len(raw_df) + len(utk_dropped)), len(utk_dropped))
print("\nRace counts:", {n: int(c) for n, c in zip(dataset_spec.race_names, np.bincount(raw_df.race, minlength=dataset_spec.num_races))})
print("Age-bin counts:", {n: int(c) for n, c in zip(AGE_BIN_NAMES, np.bincount(raw_df.age_bin, minlength=9))})

## Step 4 - Combine, deduplicate, near-duplicate audit, and split (`prepare_datasets`)

* Deduplicates **before** splitting via 64-bit perceptual dHash so exact duplicates never leak across splits.
* Supports both **`stratified`** splitting (`70/15/15` by `dataset x race x gender`) and **`official`** splitting (uses pre-existing `split == 'test'` / `'val'` rows for test and splits remaining rows into `train`/`val`).
* Supports optional **cross-dataset evaluation** (`eval_dataset`), automatically filtering out any exact dHash overlap with `train`/`val`.

In [ ]:
DROP_NEAR_DUPS = cfg.drop_near_dups
NEAR_DUP_MAX_DIST = cfg.near_dup_max_dist

def dhash_path(p):
    im = Image.open(p).convert("L").resize((9, 8), Image.BILINEAR)
    a = np.asarray(im, dtype=np.int16)
    return np.packbits((a[:, 1:] > a[:, :-1]).flatten()).tobytes().hex()

def compute_dhashes(paths, n_threads=8):
    with ThreadPoolExecutor(n_threads) as ex:
        return list(ex.map(dhash_path, paths))

_LUT16 = np.array([bin(i).count("1") for i in range(1 << 16)], dtype=np.uint8)

def popcount64(x):
    x = np.asarray(x, dtype=np.uint64)
    out = _LUT16[(x & np.uint64(0xFFFF)).astype(np.intp)].copy()
    for s in (16, 32, 48):
        out += _LUT16[((x >> np.uint64(s)) & np.uint64(0xFFFF)).astype(np.intp)]
    return out

def hashes_to_u64(series):
    return np.array([int(h, 16) for h in series], dtype=np.uint64)

def min_hamming_to_train(h_eval, h_train):
    """For each eval hash: smallest Hamming distance to any train hash."""
    return np.array([int(popcount64(h_train ^ h).min()) for h in h_eval])

def near_dup_report(data, max_dist=NEAR_DUP_MAX_DIST):
    h_tr = hashes_to_u64(data["train"]["dhash"])
    out = {}
    for k in ("val", "test"):
        md_ = min_hamming_to_train(hashes_to_u64(data[k]["dhash"]), h_tr)
        out[k] = {"min_dist": md_, "n_near": int((md_ <= max_dist).sum()), "n_exact": int((md_ == 0).sum())}
    return out

def safe_strata(key, min_count):
    key = key.copy()
    counts = key.value_counts()
    key[key.map(counts) < min_count] = "rare"
    if 0 < (key == "rare").sum() < min_count:
        key[key == "rare"] = counts.index[0]
    return key

def stratified_split(df, fracs, seed):
    key = safe_strata(df.dataset.astype(str) + "_" + df.race.astype(str) + "_" + df.gender.astype(str), 6)
    n_tr, n_va = int(round(fracs[0] * len(df))), int(round(fracs[1] * len(df)))
    tr, rest = train_test_split(df, train_size=n_tr, stratify=key, random_state=seed)
    va, te = train_test_split(rest, train_size=n_va, stratify=safe_strata(key.loc[rest.index], 4), random_state=seed)
    return [d.reset_index(drop=True) for d in (tr, va, te)]

def split_deduped_df(df: pd.DataFrame, c: Config) -> Dict[str, pd.DataFrame]:
    if c.split_mode == "official" and "split" in df.columns and (df["split"].isin(["test", "val"])).any():
        te = df[df["split"].isin(["test", "val"])].reset_index(drop=True)
        pool = df[~df["split"].isin(["test", "val"])].reset_index(drop=True)
        key = safe_strata(pool.dataset.astype(str) + "_" + pool.race.astype(str) + "_" + pool.gender.astype(str), 4)
        tr, va = train_test_split(pool, test_size=0.15, stratify=key, random_state=c.seed)
        return {"train": tr.reset_index(drop=True), "val": va.reset_index(drop=True), "test": te}
    fracs = (2/3, 1/6, 1/6) if c.quick_run else (0.7, 0.15, 0.15)
    tr, va, te = stratified_split(df, fracs, c.seed)
    return {"train": tr, "val": va, "test": te}

def prepare_datasets(c: Config, verbose: bool = True) -> Dict[str, Any]:
    """Load dataset(s), deduplicate via dHash, split into train/val/test, and optionally attach eval_dataset."""
    c = sync_config_schema(c)
    df, spec, meta = load_dataset_by_name(c.dataset, c)
    c.race_names = spec.race_names
    c.n_race = spec.num_races

    if c.quick_run:
        df = df.sample(n=min(c.quick_total, len(df)), random_state=c.seed).reset_index(drop=True)

    t0 = time.time()
    df["dhash"] = compute_dhashes(df.path.tolist())
    dup = df.duplicated("dhash", keep="first")
    n_dups = int(dup.sum())
    df = df[~dup].reset_index(drop=True)
    if verbose:
        print(f"[{spec.name}] hashed {len(df) + n_dups} images in {time.time()-t0:.1f}s; removed {n_dups} duplicates")

    splits = split_deduped_df(df, c)

    # Optional cross-dataset evaluation override
    if c.eval_dataset:
        ev_df, ev_spec, _ = load_dataset_by_name(c.eval_dataset, c)
        if ev_spec.race_names != spec.race_names:
            raise ValueError(
                f"Cross-dataset eval requires matching race schemas: train has {spec.race_names} "
                f"but eval_dataset={c.eval_dataset!r} has {ev_spec.race_names}. Use race_schema='unified_5'."
            )
        if c.quick_run:
            ev_df = ev_df.sample(n=min(max(600, c.quick_total // 3), len(ev_df)), random_state=c.seed + 99).reset_index(drop=True)
        ev_df["dhash"] = compute_dhashes(ev_df.path.tolist())
        ev_df = ev_df[~ev_df.duplicated("dhash", keep="first")].reset_index(drop=True)
        seen_hashes = set(splits["train"]["dhash"]) | set(splits["val"]["dhash"])
        ev_df = ev_df[~ev_df["dhash"].isin(seen_hashes)].reset_index(drop=True)
        if c.split_mode == "official" and "split" in ev_df.columns and (ev_df["split"].isin(["test", "val"])).any():
            ev_test = ev_df[ev_df["split"].isin(["test", "val"])].reset_index(drop=True)
        else:
            ev_test = split_deduped_df(ev_df, c)["test"]
        splits["test"] = ev_test
        if verbose:
            print(f"[cross-eval] replaced test split with {len(ev_test)} deduplicated images from {c.eval_dataset!r}")

    nd = near_dup_report(splits, max_dist=c.near_dup_max_dist)
    if verbose:
        for k, v in nd.items():
            print(f"{k}: {v['n_near']} of {len(v['min_dist'])} images within Hamming <= {c.near_dup_max_dist} of train "
                  f"({v['n_exact']} exact)")
    if c.drop_near_dups:
        splits = {
            "train": splits["train"],
            "val": splits["val"][nd["val"]["min_dist"] > c.near_dup_max_dist].reset_index(drop=True),
            "test": splits["test"][nd["test"]["min_dist"] > c.near_dup_max_dist].reset_index(drop=True),
        }
        if verbose:
            print("drop_near_dups = True -> new sizes:", {k: len(v) for k, v in splits.items()})

    if verbose:
        print("Split sizes:", {k: len(v) for k, v in splits.items()})

    return {
        "train": splits["train"],
        "val": splits["val"],
        "test": splits["test"],
        "all_df": df,
        "spec": spec,
        "near_dup": nd,
        "load_meta": meta,
        "n_duplicates_removed": n_dups,
    }

prep_out = prepare_datasets(cfg, verbose=True)
all_df = prep_out["all_df"]
dataset_spec = prep_out["spec"]
RACE_NAMES = list(dataset_spec.race_names)
data = {"train": prep_out["train"], "val": prep_out["val"], "test": prep_out["test"]}
nd = prep_out["near_dup"]
fracs = (2/3, 1/6, 1/6) if cfg.quick_run else (0.7, 0.15, 0.15)

**Validation - Step 4**

In [ ]:
tr, va, te = data["train"], data["val"], data["test"]
check("splits are non-empty", min(len(tr), len(va), len(te)) > 0, {k: len(v) for k, v in data.items()})
if not cfg.drop_near_dups and not cfg.eval_dataset:
    check("split sizes sum to the deduplicated total", len(tr) + len(va) + len(te) == len(all_df))
check("no path overlap between splits",
      not (set(tr.path) & set(va.path) or set(tr.path) & set(te.path) or set(va.path) & set(te.path)))
check("no dHash overlap between splits (no duplicate leakage)",
      not (set(tr.dhash) & set(va.dhash) or set(tr.dhash) & set(te.dhash) or set(va.dhash) & set(te.dhash)))
for name, d in data.items():
    check(f"{name}: all {dataset_spec.num_races} races and both genders present",
          d.race.nunique() == dataset_spec.num_races and d.gender.nunique() == 2)
if not cfg.eval_dataset and cfg.split_mode == "stratified":
    overall = all_df.race.value_counts(normalize=True).sort_index()
    for name, d in data.items():
        tol = 0.04 if len(d) >= 1000 else 0.10
        diff = (d.race.value_counts(normalize=True).sort_index() - overall).abs().max()
        check(f"{name}: race shares within {tol:.0%} of overall (stratified)", diff <= tol, f"max diff {diff:.3f}")
check("split is reproducible",
      split_deduped_df(all_df, cfg)["train"].path.tolist() == tr.path.tolist())

# near-duplicate Hamming distance checks
rng_ = np.random.default_rng(0)
base = rng_.integers(0, 2**63, 200, dtype=np.uint64)
check("popcount64 matches Python bit counts", all(int(popcount64(np.array([v]))[0]) == bin(int(v)).count("1") for v in base[:50]))
flip = np.uint64(1 << 3) | np.uint64(1 << 40)
d_ham = min_hamming_to_train(base[:20] ^ flip, base)
check("planted 2-bit near-duplicates are found at distance 2", bool((d_ham == 2).all()), d_ham[:5])
far = rng_.integers(0, 2**63, 20, dtype=np.uint64)
check("random hashes are far from the set (> 4)", bool((min_hamming_to_train(far, base) > 4).all()))
check("near-duplicate report covers val and test with 0 exact duplicates",
      set(nd) == {"val", "test"} and nd["val"]["n_exact"] == 0 and nd["test"]["n_exact"] == 0)
print(pd.DataFrame({k: v.race.map(dict(enumerate(RACE_NAMES))).value_counts() for k, v in data.items()}))

## Step 5 - Image cache

Images are decoded **once** into a `uint8` tensor `(N x 3 x S x S)` and kept on the GPU; augmentation then runs on the GPU. The cache is saved as `.npy` keyed by `(paths, size)` so subsequent `run_experiment` calls on the same split skip disk decoding.

In [ ]:
def load_resized(p, size):
    im = Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR)
    return np.asarray(im, dtype=np.uint8)

def build_cache(paths, size, cache_dir=None, n_threads=8):
    """paths -> uint8 tensor (N, 3, size, size). Cached on disk by a hash of (paths, size)."""
    paths = list(paths)
    f = None
    if cache_dir:
        Path(cache_dir).mkdir(parents=True, exist_ok=True)
        key = hashlib.md5(("|".join(paths) + f"|{size}").encode()).hexdigest()[:16]
        f = Path(cache_dir) / f"cache_{key}.npy"
        if f.exists():
            return torch.from_numpy(np.load(f)).permute(0, 3, 1, 2).contiguous()
    with ThreadPoolExecutor(n_threads) as ex:
        arr = np.stack(list(ex.map(lambda p: load_resized(p, size), paths)))
    if f is not None:
        np.save(f, arr)
    return torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous()

def to_device(cache_cpu, df):
    lab = torch.tensor(df[["gender", "race", "age_bin"]].values, dtype=torch.long, device=DEVICE)
    return cache_cpu.to(DEVICE), lab

t0 = time.time()
train_cache_cpu = build_cache(data["train"].path, cfg.img_size, cfg.cache_dir)
print(f"train cache {tuple(train_cache_cpu.shape)} built in {time.time()-t0:.1f}s,",
      f"{train_cache_cpu.numel()/1e6:.0f} MB")

**Validation - Step 5**

In [ ]:
c = train_cache_cpu
check("cache shape is (N, 3, S, S)", c.shape == (len(data["train"]), 3, cfg.img_size, cfg.img_size), tuple(c.shape))
check("cache dtype is uint8", c.dtype == torch.uint8)
flat_std = c.float().flatten(1).std(1)
check("no blank / constant images", bool((flat_std > 1.0).all()), f"{int((flat_std <= 1.0).sum())} suspicious")
t0 = time.time(); again = build_cache(data["train"].path, cfg.img_size, cfg.cache_dir)
check("reload from disk gives identical tensor", torch.equal(c, again), f"{time.time()-t0:.2f}s")
if cfg.show_plots:
    fig, axs = plt.subplots(1, 8, figsize=(16, 2.4))
    for k, ax in enumerate(axs):
        r = data["train"].iloc[k]
        ax.imshow(c[k].permute(1, 2, 0).numpy()); ax.axis("off")
        ax.set_title(f"{GENDER_NAMES[r.gender][0]} {RACE_NAMES[r.race][:7]}\n{AGE_BIN_NAMES[r.age_bin]}", fontsize=8)
    plt.show()
print("Visual check: faces should be centred, upright, labels plausible.")

## Step 6 - GPU augmentation (mild & strong, per-sample randomness)

Draws an independent flip, rotation, scale, shift, brightness, and contrast **per image**:
* `augment` / `prep`: mild augmentation for the attribute-pair branch.
* `strong_augment` / `instance_views`: stronger affine + brightness/contrast + occasional grayscale for the SimCLR-style instance branch.

In [ ]:
def augment(x, strength=1.0, flip_p=0.5):
    """x: float (B,3,H,W) in [0,1]. Per-sample affine + flip + brightness/contrast."""
    B, dev = x.shape[0], x.device
    u = lambda: torch.rand(B, device=dev) * 2 - 1
    ang = u() * math.radians(10) * strength
    sc = 1.0 + u() * 0.10 * strength
    tx, ty = u() * 0.10 * strength, u() * 0.10 * strength
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    flip = torch.rand(B, device=dev) < flip_p
    x = torch.where(flip.view(B, 1, 1, 1), x.flip(3), x)
    br = 1.0 + u().view(B, 1, 1, 1) * 0.2 * strength
    ct = 1.0 + u().view(B, 1, 1, 1) * 0.2 * strength
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    return (((x - m) * ct + m) * br).clamp(0, 1)

def strong_augment(x, strength=2.5, gray_p=0.2):
    """Strong per-sample augmentation for instance views. x: float (B,3,H,W) in [0,1]."""
    x = augment(x, strength=strength)
    g = (torch.rand(x.shape[0], device=x.device) < gray_p).view(-1, 1, 1, 1)
    return torch.where(g, x.mean(1, keepdim=True).expand_as(x), x)

def prep(x_uint8, train, strength=1.0):
    x = x_uint8.float() / 255.0
    if train:
        x = augment(x, strength)
    return (x - 0.5) / 0.5

def instance_views(cache, idx, cfg):
    base = cache[idx].float() / 255.0
    return [(strong_augment(base, cfg.inst_strength) - 0.5) / 0.5 for _ in range(2)]

**Validation - Step 6**

In [ ]:
x = (train_cache_cpu[:1].float() / 255.0).repeat(16, 1, 1, 1).to(DEVICE)
y = augment(x)
ys = strong_augment(x, cfg.inst_strength)
check("augment keeps shape and [0,1] range", y.shape == x.shape and float(y.min()) >= 0 and float(y.max()) <= 1)
check("augment is random PER SAMPLE (same image -> different outputs)", float(y.std(0).mean()) > 0.01, f"{float(y.std(0).mean()):.4f}")
ident = augment(x, strength=0.0, flip_p=0.0)
check("strength 0 + no flip is the identity", float((ident - x).abs().max()) < 1e-4, f"{float((ident - x).abs().max()):.2e}")
check("strong_augment keeps shape and [0,1] range", ys.shape == x.shape and float(ys.min()) >= 0 and float(ys.max()) <= 1)
check("strong views vary more than mild views", float(ys.std(0).mean()) > float(y.std(0).mean()),
      f"{float(ys.std(0).mean()):.3f} vs {float(y.std(0).mean()):.3f}")
n = prep(train_cache_cpu[:8].to(DEVICE), train=False)
check("prep output is in [-1, 1]", float(n.min()) >= -1.0001 and float(n.max()) <= 1.0001)
if cfg.show_plots:
    fig, axs = plt.subplots(3, 8, figsize=(16, 6.2))
    for k in range(8):
        axs[0, k].imshow(x[k].permute(1, 2, 0).cpu()); axs[0, k].axis("off")
        axs[1, k].imshow(y[k].permute(1, 2, 0).cpu()); axs[1, k].axis("off")
        axs[2, k].imshow(ys[k].permute(1, 2, 0).cpu()); axs[2, k].axis("off")
    axs[0, 0].set_title("original", fontsize=8); axs[1, 0].set_title("mild aug", fontsize=8); axs[2, 0].set_title("strong aug", fontsize=8)
    plt.show()

## Step 7 - Pair sampler (attribute pairs)

Constructs balanced positive/negative demographic pairs with dynamic support for any `n_race` x `n_age` grid (e.g., 5-class `unified_5` or 7-class FairFace `native`), adjacent-age-bin exclusion, and difficulty tracking (`ndiff = 1, 2, 3`).

In [ ]:
class PairSampler:
    def __init__(self, df, group_power=0.5, n_race=None, n_age=9):
        self.g = df.gender.values.astype(int)
        self.r = df.race.values.astype(int)
        self.a = df.age_bin.values.astype(int)
        self.n = len(df)
        self.n_race = int(n_race) if n_race is not None else max(5, int(self.r.max()) + 1)
        self.n_age = int(n_age)
        gid = self.g * (self.n_race * self.n_age) + self.r * self.n_age + self.a
        self.members = {k: np.flatnonzero(gid == k) for k in np.unique(gid)}
        keys = np.array(sorted(self.members))
        sizes = np.array([len(self.members[k]) for k in keys], float)
        self.all_groups, self.all_p = keys, sizes ** group_power / (sizes ** group_power).sum()
        pos = sizes >= 2
        self.pos_groups = keys[pos]
        self.pos_p = sizes[pos] ** group_power / (sizes[pos] ** group_power).sum()

    def ndiff(self, i, j):
        return int(self.g[i] != self.g[j]) + int(self.r[i] != self.r[j]) + int(abs(self.a[i] - self.a[j]) >= 2)

    def _positive(self, rng):
        k = rng.choice(self.pos_groups, p=self.pos_p)
        i, j = rng.choice(self.members[k], 2, replace=False)
        return int(i), int(j), 0

    def _negative(self, rng, target=None):
        k = rng.choice(self.all_groups, p=self.all_p)
        i = int(rng.choice(self.members[k]))
        for want in ([target, None] if target else [None]):
            for _ in range(300):
                j = int(rng.integers(self.n))
                nd = self.ndiff(i, j)
                if nd >= 1 and (want is None or nd == want):
                    return i, j, nd
        raise RuntimeError("could not find a valid negative")

    def sample(self, n, rng, stratify_neg=False):
        pos = rng.permutation(np.arange(n) % 2 == 0)        # exactly 50% positives
        targets = rng.permutation(np.arange(n) % 3 + 1)      # 1, 2, 3 cycle for negatives
        out = [self._positive(rng) if p else self._negative(rng, int(t) if stratify_neg else None)
               for p, t in zip(pos, targets)]
        i, j, nd = (np.array(v) for v in zip(*out))
        return {"i": i, "j": j, "y": pos.astype(np.float32), "ndiff": nd}

    def make_fixed(self, n, seed):
        return self.sample(n, np.random.default_rng(seed), stratify_neg=True)

train_sampler = PairSampler(data["train"], cfg.group_power, n_race=cfg.n_race, n_age=cfg.n_age)
print(f"{len(train_sampler.members)} groups ({len(train_sampler.pos_groups)} usable for positives)")

**Validation - Step 7**

In [ ]:
s = train_sampler
P = s.sample(4000, np.random.default_rng(0))
pos, neg = P["y"] == 1, P["y"] == 0
check("positive rate is 50%", abs(pos.mean() - 0.5) < 0.02, f"{pos.mean():.3f}")
same = lambda i, j: (s.g[i] == s.g[j]) & (s.r[i] == s.r[j]) & (s.a[i] == s.a[j])
check("positives share gender, race and age bin", bool(same(P["i"][pos], P["j"][pos]).all()))
check("positives are two different images", bool((P["i"][pos] != P["j"][pos]).all()))
check("negatives differ in at least one attribute", bool((P["ndiff"][neg] >= 1).all()))
adj_only = (s.g[P["i"]] == s.g[P["j"]]) & (s.r[P["i"]] == s.r[P["j"]]) & (np.abs(s.a[P["i"]] - s.a[P["j"]]) == 1)
check("no adjacent-bin-only pairs", not bool(adj_only.any()))
F1 = s.make_fixed(1000, 7); F2 = s.make_fixed(1000, 7); F3 = s.make_fixed(1000, 8)
check("fixed eval pairs are reproducible", all(np.array_equal(F1[k], F2[k]) for k in F1))
check("different seed gives different pairs", not np.array_equal(F1["i"], F3["i"]))
nd_c = np.bincount(F1["ndiff"][F1["y"] == 0], minlength=4)[1:]
check("fixed negatives stratified over ndiff 1/2/3", nd_c.min() > 0.2 * nd_c.sum(), {k: int(v) for k, v in zip([1, 2, 3], nd_c)})
print("negatives by ndiff in random training pairs:", {k: int(v) for k, v in zip([1, 2, 3], np.bincount(P["ndiff"][neg], minlength=4)[1:])})

## Step 8 - Model and losses (dual projection heads + dynamic attribute heads)

* **Backbone:** four conv blocks (~1.37M parameters) -> `feat_dim` shared features.
* **Dual projection heads:** `proj_attr` and `proj_inst` (each `feat_dim -> feat_dim -> emb_dim`, L2-normalised).
* **Dynamic attribute heads:** gender (`2`), race (`cfg.n_race`), and age bin (`cfg.n_age`).

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.MaxPool2d(2))

class SiameseCNN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        f = cfg.feat_dim
        self.backbone = nn.Sequential(conv_block(3, 32), conv_block(32, 64), conv_block(64, 128),
                                      conv_block(128, f), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        mk = lambda: nn.Sequential(nn.Linear(f, f), nn.ReLU(inplace=True), nn.Linear(f, cfg.emb_dim))
        self.proj_attr, self.proj_inst = mk(), mk()
        self.drop = nn.Dropout(0.3)
        self.gender = nn.Linear(f, 2)
        self.race = nn.Linear(f, cfg.n_race)
        self.age = nn.Linear(f, cfg.n_age)

    def forward(self, x):
        f = self.backbone(x)
        ea = F.normalize(self.proj_attr(f).float(), dim=1)
        ei = F.normalize(self.proj_inst(f).float(), dim=1)
        fd = self.drop(f)
        return ea, ei, self.gender(fd), self.race(fd), self.age(fd)

def amp_ctx(cfg):
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=cfg.amp)

def nt_xent(z1, z2, temperature):
    """z1, z2: (B, D) L2-normalised, row k of z1 and z2 are two views of the same image."""
    B = z1.shape[0]
    z = torch.cat([z1, z2])
    sim = (z @ z.t()) / temperature
    sim = sim.masked_fill(torch.eye(2 * B, dtype=torch.bool, device=z.device), float("-inf"))
    target = torch.cat([torch.arange(B, 2 * B), torch.arange(0, B)]).to(z.device)
    return F.cross_entropy(sim, target)

def compute_loss(model, xa1, xa2, y, ta1, ta2, xi1, xi2, cfg):
    """Combined attribute-pair contrastive loss, instance NT-Xent loss, and attribute head CE losses."""
    B, Bi = xa1.shape[0], xi1.shape[0]
    with amp_ctx(cfg):
        ea, ei, g, r, a = model(torch.cat([xa1, xa2, xi1, xi2]))
    d = (ea[:B] - ea[B:2 * B]).pow(2).sum(1).add(1e-8).sqrt()
    con = (y * d.pow(2) + (1 - y) * F.relu(cfg.margin - d).pow(2)).mean()
    inst = nt_xent(ei[2 * B:2 * B + Bi], ei[2 * B + Bi:], cfg.temperature)
    t, n = torch.cat([ta1, ta2]), 2 * B                      # heads: attribute-pair images only
    lg = F.cross_entropy(g[:n].float(), t[:, 0]); lr_ = F.cross_entropy(r[:n].float(), t[:, 1])
    la = F.cross_entropy(a[:n].float(), t[:, 2])
    heads = (lg + lr_ + la) / 3
    total = cfg.w_pair * con + cfg.w_inst * inst + cfg.attr_weight * heads
    parts = {"contrastive": con.detach(), "inst": inst.detach(), "heads": heads.detach(),
             "gender": lg.detach(), "race": lr_.detach(), "age": la.detach()}
    return total, parts, d.detach()

demo_model = SiameseCNN(cfg).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in demo_model.parameters()):,}")

**Validation - Step 8** (NT-Xent sanity, shapes, gradient isolation per head, single-batch overfit test)

In [ ]:
# NT-Xent sanity
set_seed(cfg.seed)
z = F.normalize(torch.randn(64, 128), dim=1)
l_same = nt_xent(z, z.clone(), 0.2).item()
l_rand = nt_xent(z, F.normalize(torch.randn(64, 128), dim=1), 0.2).item()
check("NT-Xent: identical views give a much lower loss than random views", l_same < 0.5 * l_rand, f"{l_same:.3f} vs {l_rand:.3f}")
check("NT-Xent: random loss is near log(2B-1)", abs(l_rand - math.log(127)) < 0.5, f"{l_rand:.3f} vs {math.log(127):.3f}")

# model parameter count & output shapes
set_seed(cfg.seed)
m = SiameseCNN(cfg).to(DEVICE)
n_par = sum(p.numel() for p in m.parameters())
check("parameter count is a 'small CNN' (0.8M - 2.0M)", 0.8e6 < n_par < 2.0e6, f"{n_par:,}")

x = torch.randn(6, 3, cfg.img_size, cfg.img_size, device=DEVICE)
m.eval()
with torch.no_grad():
    ea, ei, g, r, a = m(x); ea2, ei2 = m(x)[:2]
check("output shapes", ea.shape == (6, cfg.emb_dim) and ei.shape == (6, cfg.emb_dim) and g.shape == (6, 2)
      and r.shape == (6, cfg.n_race) and a.shape == (6, cfg.n_age))
check("both embeddings are unit length", torch.allclose(ea.norm(dim=1), torch.ones(6, device=DEVICE), atol=1e-4)
      and torch.allclose(ei.norm(dim=1), torch.ones(6, device=DEVICE), atol=1e-4))
check("eval mode is deterministic", torch.equal(ea, ea2) and torch.equal(ei, ei2))
check("the two projection heads are different modules with different weights",
      not torch.equal(m.proj_attr[0].weight, m.proj_inst[0].weight))

# gradient routing and isolation tests
tr_cache, tr_lab = to_device(train_cache_cpu, data["train"])
P = train_sampler.sample(32, np.random.default_rng(1))
ii_p = torch.as_tensor(P["i"], device=DEVICE); jj_p = torch.as_tensor(P["j"], device=DEVICE)
xa1, xa2, yy, ta1, ta2 = prep(tr_cache[ii_p], False), prep(tr_cache[jj_p], False), torch.as_tensor(P["y"], device=DEVICE), tr_lab[ii_p], tr_lab[jj_p]
ii_inst = torch.as_tensor(np.random.default_rng(2).choice(len(tr_cache), 32, replace=False), device=DEVICE)
xi1, xi2 = instance_views(tr_cache, ii_inst, cfg)
cfg32 = dataclasses.replace(cfg, amp=False)

def grad_norms(w_pair, w_inst, attr_w):
    mm = SiameseCNN(cfg).to(DEVICE); mm.train()
    c = dataclasses.replace(cfg32, w_pair=w_pair, w_inst=w_inst, attr_weight=attr_w)
    loss, _, _ = compute_loss(mm, xa1, xa2, yy, ta1, ta2, xi1, xi2, c)
    loss.backward()
    gn = lambda mod: sum(float(p.grad.abs().sum()) for p in mod.parameters() if p.grad is not None)
    return {"proj_attr": gn(mm.proj_attr), "proj_inst": gn(mm.proj_inst), "heads": gn(mm.gender) + gn(mm.race) + gn(mm.age),
            "backbone": gn(mm.backbone)}

full = grad_norms(1, 1, 1)
check("full loss: gradients reach backbone, both projection heads and the attribute heads", all(v > 0 for v in full.values()), full)
no_inst = grad_norms(1, 0, 1)
check("w_inst = 0: instance projection head gets no gradient", no_inst["proj_inst"] == 0 and no_inst["proj_attr"] > 0, no_inst)
no_pair = grad_norms(0, 1, 1)
check("w_pair = 0: attribute projection head gets no gradient", no_pair["proj_attr"] == 0 and no_pair["proj_inst"] > 0, no_pair)
heads_off = grad_norms(1, 1, 0)
check("attr_weight = 0: attribute heads get no gradient", heads_off["heads"] == 0, heads_off)

# overfit one fixed batch (fp32)
m = SiameseCNN(cfg).to(DEVICE); m.train()
opt = torch.optim.AdamW(m.parameters(), lr=2e-3)
losses = []
for step in range(100):
    loss, _, _ = compute_loss(m, xa1, xa2, yy, ta1, ta2, xi1, xi2, cfg32)
    opt.zero_grad(); loss.backward()
    if step == 0:
        no_grad = [n for n, p in m.named_parameters() if p.grad is None or not torch.isfinite(p.grad).all()]
        check("gradients reach every parameter (finite)", not no_grad, no_grad[:3])
    opt.step(); losses.append(loss.item())
check("loss is finite throughout", all(math.isfinite(v) for v in losses))
check("overfit test: loss drops below 60% of start", losses[-1] < 0.6 * losses[0], f"{losses[0]:.3f} -> {losses[-1]:.3f}")
del m, opt, demo_model

## Step 9 - Training loop and evaluation helpers

* **Training (`fit`):** AdamW + per-iteration OneCycleLR schedule, mixed precision, gradient clipping, automatic `steps_per_epoch` resolution (`int` or `None` for full-dataset epoch), early stopping, and `last.pt` / `best.pt` checkpointing.
* **Verification & classification:** attribute-pair verification, instance verification, full multi-class reports (with dynamic `race_names`), linear probes, image-level bootstrap CIs, and balanced-group fairness.

In [ ]:
def make_scaler(enabled):
    try:
        from torch.amp import GradScaler
        return GradScaler("cuda", enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)

def _fork():
    return torch.random.fork_rng(devices=[torch.cuda.current_device()] if DEVICE.type == "cuda" else [])

def resolve_steps_per_epoch(cfg, n_train: int) -> int:
    if cfg.steps_per_epoch is not None:
        return int(cfg.steps_per_epoch)
    return max(1, int(n_train) // max(1, 2 * int(cfg.batch_pairs)))

def get_pair_batch(cache, lab, pairs, lo, hi, train):
    i = torch.as_tensor(pairs["i"][lo:hi], device=DEVICE, dtype=torch.long)
    j = torch.as_tensor(pairs["j"][lo:hi], device=DEVICE, dtype=torch.long)
    y = torch.as_tensor(pairs["y"][lo:hi], device=DEVICE, dtype=torch.float32)
    return prep(cache[i], train), prep(cache[j], train), y, lab[i], lab[j]

@torch.no_grad()
def evaluate(model, cache, lab, pairs, cfg, bs=256):
    """Validation objective, attribute-pair distances, and epoch classification/verification metrics."""
    model.eval()
    con_s, head_s, n_pairs, dists = 0.0, 0.0, 0, []
    pg_l, pr_l, pa_l, tg_l, tr_l, ta_l = [], [], [], [], [], []
    for lo in range(0, len(pairs["y"]), bs):
        x1, x2, y, t1, t2 = get_pair_batch(cache, lab, pairs, lo, lo + bs, False)
        B = len(y)
        with amp_ctx(cfg):
            ea, _, g, r, a = model(torch.cat([x1, x2]))
        d = (ea[:B] - ea[B:]).pow(2).sum(1).add(1e-8).sqrt()
        con_s += float((y * d.pow(2) + (1 - y) * F.relu(cfg.margin - d).pow(2)).sum())
        t = torch.cat([t1, t2])
        head_s += float((F.cross_entropy(g.float(), t[:, 0], reduction="sum") + F.cross_entropy(r.float(), t[:, 1], reduction="sum")
                         + F.cross_entropy(a.float(), t[:, 2], reduction="sum")) / 3)
        n_pairs += B; dists.append(d.float().cpu())
        pg_l.append(g.argmax(1).cpu()); pr_l.append(r.argmax(1).cpu()); pa_l.append(a.argmax(1).cpu())
        tg_l.append(t[:, 0].cpu()); tr_l.append(t[:, 1].cpu()); ta_l.append(t[:, 2].cpu())
    m_ = min(256, len(cache))
    with _fork():
        torch.manual_seed(cfg.eval_seed)
        xi1, xi2 = instance_views(cache, torch.arange(m_, device=DEVICE), cfg)
        with amp_ctx(cfg):
            ei = model(torch.cat([xi1, xi2]))[1]
        inst = float(nt_xent(ei[:m_], ei[m_:], cfg.temperature))
    con, heads = con_s / n_pairs, head_s / (2 * n_pairs)

    d_np = torch.cat(dists).numpy()
    y_np = pairs["y"].astype(int)
    v_pred = (d_np <= 0.5 * cfg.margin).astype(int)
    vp, vr, vf1, _ = precision_recall_fscore_support(y_np, v_pred, average="binary", zero_division=0)
    parts = {
        "contrastive": con, "inst": inst, "heads": heads,
        "attr_acc": float((v_pred == y_np).mean()), "attr_prec": float(vp),
        "attr_rec": float(vr), "attr_f1": float(vf1),
    }
    for attr_name, preds_t, trues_t in (("gender", pg_l, tg_l), ("race", pr_l, tr_l), ("age", pa_l, ta_l)):
        yp, yt = torch.cat(preds_t).numpy(), torch.cat(trues_t).numpy()
        mp, mr, mf1, _ = precision_recall_fscore_support(yt, yp, labels=np.unique(yt), average="macro", zero_division=0)
        parts[f"{attr_name}_acc"] = float((yp == yt).mean())
        parts[f"{attr_name}_prec"] = float(mp)
        parts[f"{attr_name}_rec"] = float(mr)
        parts[f"{attr_name}_f1"] = float(mf1)

    total = cfg.w_pair * con + cfg.w_inst * inst + cfg.attr_weight * heads
    return total, parts, d_np, y_np

def fit(model, tr_cache, tr_lab, tr_sampler, va_cache, va_lab, va_pairs, cfg, out_dir, stop_after=None, verbose=True):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    model.to(DEVICE)
    n_tr = len(tr_cache)
    eff_steps = resolve_steps_per_epoch(cfg, n_tr)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=cfg.lr, total_steps=max(1, cfg.epochs * eff_steps), pct_start=0.15
    )
    scaler = make_scaler(cfg.amp)
    history, best, start, epochs_no_improve = [], float("inf"), 0, 0

    if cfg.resume and (out / "last.pt").exists():
        ck = torch.load(out / "last.pt", map_location=DEVICE)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        history, best, start = ck["history"], ck["best"], ck["epoch"]
        epochs_no_improve = ck.get("epochs_no_improve", 0)
        if verbose:
            print(f"resumed from epoch {start} (best val {best:.3f}, no_improve {epochs_no_improve})")

    rng = np.random.default_rng(cfg.seed + start)
    patience = getattr(cfg, "early_stop_patience", 0)
    min_delta = getattr(cfg, "early_stop_min_delta", 0.0)

    val_metric_keys = (
        "contrastive", "inst", "heads",
        "attr_acc", "attr_prec", "attr_rec", "attr_f1",
        "gender_acc", "gender_prec", "gender_rec", "gender_f1",
        "race_acc", "race_prec", "race_rec", "race_f1",
        "age_acc", "age_prec", "age_rec", "age_f1",
    )

    for ep in range(start, cfg.epochs):
        model.train(); t0 = time.time(); acc = Counter()
        for _ in range(eff_steps):
            pairs = tr_sampler.sample(cfg.batch_pairs, rng)
            xa1, xa2, y, ta1, ta2 = get_pair_batch(tr_cache, tr_lab, pairs, 0, cfg.batch_pairs, True)
            ii = torch.as_tensor(rng.choice(n_tr, min(cfg.batch_inst, n_tr), replace=False), device=DEVICE)
            xi1, xi2 = instance_views(tr_cache, ii, cfg)
            loss, parts, _ = compute_loss(model, xa1, xa2, y, ta1, ta2, xi1, xi2, cfg)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            acc["loss"] += loss.item()
            for k, v in parts.items():
                acc[k] += float(v)

        va_total, va_parts, vd, vy = evaluate(model, va_cache, va_lab, va_pairs, cfg)
        rec = {
            "epoch": ep + 1, "steps": eff_steps,
            "train_loss": acc["loss"] / eff_steps, "val_loss": va_total,
            "val_auc_attr": float(roc_auc_score(vy, -vd)),
            "val_pr_auc_attr": float(average_precision_score(vy, -vd)),
            **{f"train_{k}": acc[k] / eff_steps for k in ("contrastive", "inst", "heads")},
            **{f"val_{k}": va_parts[k] for k in val_metric_keys},
            "lr": sched.get_last_lr()[0], "seconds": time.time() - t0,
        }
        history.append(rec)

        if va_total < best - min_delta:
            best = va_total
            epochs_no_improve = 0
            torch.save({"model": model.state_dict(), "epoch": ep + 1, "val_loss": va_total}, out / "best.pt")
        else:
            epochs_no_improve += 1

        torch.save({
            "model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
            "scaler": scaler.state_dict(), "history": history, "epoch": ep + 1,
            "best": best, "epochs_no_improve": epochs_no_improve,
        }, out / "last.pt")

        if verbose:
            print(
                f"[{cfg.name} s{cfg.seed}] epoch {ep+1}/{cfg.epochs} ({eff_steps} steps) | train {rec['train_loss']:.3f} | val {va_total:.3f} | "
                f"attr-AUC {rec['val_auc_attr']:.3f} F1 {rec['val_attr_f1']:.3f} | "
                f"acc(G/R/A) {rec['val_gender_acc']:.2f}/{rec['val_race_acc']:.2f}/{rec['val_age_acc']:.2f} | "
                f"patience {epochs_no_improve}/{patience or '-'} | {rec['seconds']:.0f}s"
            )

        if stop_after is not None and ep + 1 >= stop_after:
            break
        if patience and epochs_no_improve >= patience:
            if verbose:
                print(f"[{cfg.name} s{cfg.seed}] Early stopping at epoch {ep+1} (best val_loss {best:.3f})")
            break

    return history

@torch.no_grad()
def predict_all(model, cache, cfg, bs=512):
    model.eval()
    EA, EI, G, R, A = [], [], [], [], []
    for lo in range(0, len(cache), bs):
        with amp_ctx(cfg):
            ea, ei, g, r, a = model(prep(cache[lo:lo + bs].to(DEVICE), False))
        EA.append(ea.float().cpu()); EI.append(ei.float().cpu())
        G.append(F.softmax(g.float(), 1).cpu()); R.append(F.softmax(r.float(), 1).cpu()); A.append(F.softmax(a.float(), 1).cpu())
    c = lambda L: torch.cat(L).numpy()
    return {"emb_attr": c(EA), "emb_inst": c(EI), "p_gender": c(G), "p_race": c(R), "p_age": c(A)}

@torch.no_grad()
def instance_eval(model, cache, cfg, seed, n=None):
    """Positive: image vs a seeded strong augmentation of itself. Negative: two different images."""
    model.eval()
    N = len(cache); n = min(n or cfg.n_inst_eval, N)
    rng = np.random.default_rng(seed)
    idx = rng.choice(N, n, replace=False)
    shift = int(rng.integers(1, n))
    neg_idx = np.roll(idx, shift)
    Ec, Ea = [], []
    with _fork():
        torch.manual_seed(seed)
        for lo in range(0, n, 256):
            xb = cache[torch.as_tensor(idx[lo:lo + 256], device=DEVICE)].float() / 255.0
            xa = strong_augment(xb, cfg.inst_strength)
            with amp_ctx(cfg):
                Ec.append(model((xb - 0.5) / 0.5)[1].float().cpu()); Ea.append(model((xa - 0.5) / 0.5)[1].float().cpu())
    Ec, Ea = torch.cat(Ec).numpy(), torch.cat(Ea).numpy()
    dpos = np.linalg.norm(Ec - Ea, axis=1)
    dneg = np.linalg.norm(Ec - np.roll(Ec, shift, axis=0), axis=1)
    return {"dist": np.concatenate([dpos, dneg]), "y": np.r_[np.ones(n, int), np.zeros(n, int)],
            "i": np.concatenate([idx, idx]), "j": np.concatenate([idx, neg_idx]), "ndiff": np.zeros(2 * n, int)}

def best_threshold(dist, y):
    cands = np.unique(np.quantile(dist, np.linspace(0, 1, 201)))
    accs = [((dist <= t).astype(int) == y).mean() for t in cands]
    return float(cands[int(np.argmax(accs))])

def verification_report(dist, y, ndiff, thr):
    fpr, tpr, _ = roc_curve(y, -dist)
    fnr = 1 - tpr; k = int(np.nanargmin(np.abs(fnr - fpr)))
    pr_c, rc_c, _ = precision_recall_curve(y, -dist)
    pred = (dist <= thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    spec = float(tn / max(tn + fp, 1))
    npv = float(tn / max(tn + fn, 1))
    bal_acc = float(0.5 * (r + spec))
    mcc = float(matthews_corrcoef(y, pred)) if len(np.unique(pred)) > 1 and len(np.unique(y)) > 1 else 0.0

    by_nd, metrics_by_nd = {}, {}
    for nd in (1, 2, 3):
        m = (y == 1) | (ndiff == nd)
        if (ndiff[m] == nd).any():
            by_nd[nd] = float(roc_auc_score(y[m], -dist[m]))
            p_nd, r_nd, f1_nd, _ = precision_recall_fscore_support(y[m], pred[m], average="binary", zero_division=0)
            metrics_by_nd[nd] = {
                "roc_auc": by_nd[nd],
                "pr_auc": float(average_precision_score(y[m], -dist[m])),
                "accuracy": float((pred[m] == y[m]).mean()),
                "precision": float(p_nd),
                "recall": float(r_nd),
                "f1": float(f1_nd),
            }
    step = max(1, len(fpr) // 200)
    p_step = max(1, len(pr_c) // 200)
    return {
        "roc_auc": float(roc_auc_score(y, -dist)),
        "pr_auc": float(average_precision_score(y, -dist)),
        "eer": float((fpr[k] + fnr[k]) / 2),
        "accuracy": float((pred == y).mean()),
        "balanced_accuracy": bal_acc,
        "precision": float(p),
        "recall": float(r),
        "specificity": spec,
        "npv": npv,
        "f1": float(f1),
        "mcc": mcc,
        "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]],
        "threshold": thr,
        "positive_rate": float(y.mean()),
        "n_pairs": int(len(y)),
        "auc_by_ndiff": by_nd,
        "metrics_by_ndiff": metrics_by_nd,
        "roc": {"fpr": fpr[::step].tolist(), "tpr": tpr[::step].tolist()},
        "pr_curve": {"precision": pr_c[::p_step].tolist(), "recall": rc_c[::p_step].tolist()},
    }

def cls_report(y, p, k, names, probs=None):
    present = np.unique(y)
    labels = list(range(k))
    pr_macro, rc_macro, f1_macro, _ = precision_recall_fscore_support(y, p, labels=present, average="macro", zero_division=0)
    pr_w, rc_w, f1_w, _ = precision_recall_fscore_support(y, p, labels=present, average="weighted", zero_division=0)
    pr_micro, rc_micro, f1_micro, _ = precision_recall_fscore_support(y, p, labels=present, average="micro", zero_division=0)
    pr_c, rc_c, f1_c, sup_c = precision_recall_fscore_support(y, p, labels=labels, average=None, zero_division=0)
    cm = confusion_matrix(y, p, labels=labels)
    per_class = {
        names[c]: {
            "precision": float(pr_c[c]),
            "recall": float(rc_c[c]),
            "f1": float(f1_c[c]),
            "support": int(sup_c[c]),
        }
        for c in range(k)
    }
    out = {
        "accuracy": float(accuracy_score(y, p)),
        "balanced_accuracy": float(balanced_accuracy_score(y, p)),
        "precision_macro": float(pr_macro),
        "recall_macro": float(rc_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(pr_w),
        "recall_weighted": float(rc_w),
        "f1_weighted": float(f1_w),
        "precision_micro": float(pr_micro),
        "recall_micro": float(rc_micro),
        "f1_micro": float(f1_micro),
        "mcc": float(matthews_corrcoef(y, p)) if len(np.unique(p)) > 1 and len(present) > 1 else 0.0,
        "cohen_kappa": float(cohen_kappa_score(y, p)) if len(present) > 1 else 0.0,
        "per_class": per_class,
        "confusion_matrix": cm.tolist(),
        "class_names": list(names),
        "classification_report": classification_report(y, p, labels=labels, target_names=list(names), digits=4, zero_division=0),
    }
    if probs is not None:
        probs = np.asarray(probs, dtype=np.float64)
        probs_clip = np.clip(probs, 1e-7, 1.0 - 1e-7)
        probs_clip = probs_clip / probs_clip.sum(axis=1, keepdims=True)
        out["log_loss"] = float(log_loss(y, probs_clip, labels=labels))
        if len(present) == k and k > 1:
            if k == 2:
                out["roc_auc_ovr_macro"] = float(roc_auc_score(y, probs_clip[:, 1]))
                out["roc_auc_ovr_weighted"] = out["roc_auc_ovr_macro"]
                out["pr_auc_macro"] = float(average_precision_score(y, probs_clip[:, 1]))
            else:
                out["roc_auc_ovr_macro"] = float(roc_auc_score(y, probs_clip, multi_class="ovr", average="macro", labels=labels))
                out["roc_auc_ovr_weighted"] = float(roc_auc_score(y, probs_clip, multi_class="ovr", average="weighted", labels=labels))
                y_oh = np.eye(k)[y]
                out["pr_auc_macro"] = float(average_precision_score(y_oh, probs_clip, average="macro"))
        else:
            out["roc_auc_ovr_macro"] = float("nan")
            out["roc_auc_ovr_weighted"] = float("nan")
            out["pr_auc_macro"] = float("nan")
    return out

def fairness_report(df, pg, pr, pa, min_n=30, race_names=None):
    r_names = list(race_names) if race_names is not None else RACE_NAMES
    g, r, a = df.gender.values, df.race.values, df.age_bin.values
    stat = lambda m: {"n": int(m.sum()), "gender_acc": float((pg[m] == g[m]).mean()),
                      "race_acc": float((pr[m] == r[m]).mean()), "age_bin_acc": float((pa[m] == a[m]).mean())}
    out = {"by_race": {r_names[k]: stat(r == k) for k in np.unique(r)},
           "by_gender": {GENDER_NAMES[k]: stat(g == k) for k in np.unique(g)},
           "by_race_gender": {f"{r_names[k]}/{GENDER_NAMES[q]}": stat((r == k) & (g == q))
                              for k in np.unique(r) for q in np.unique(g)}}
    out["summary"] = {}
    for view in ("by_race", "by_race_gender"):
        ok = {k: v for k, v in out[view].items() if v["n"] >= min_n}
        for metric in ("gender_acc", "race_acc", "age_bin_acc"):
            vals = [v[metric] for v in ok.values()]
            if vals:
                out["summary"][f"{view}.{metric}"] = {"worst": float(min(vals)), "best": float(max(vals)),
                                                      "gap": float(max(vals) - min(vals)), "groups_used": len(vals)}
    return out

def boot_counts(n, B, seed):
    """B x n multinomial resampling counts (how often each image is drawn)."""
    return np.random.default_rng(seed).multinomial(n, np.full(n, 1.0 / n), size=B).astype(np.float32)

def ci(x):
    return [float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))]

def boot_accuracy(correct, counts):
    return (counts @ correct.astype(np.float32)) / counts.sum(1)

def boot_group(correct, masks, counts, min_n=30):
    """Per-group CIs plus CIs for worst-group accuracy and the best-minus-worst gap."""
    c = correct.astype(np.float32)
    accs = {g: (counts[:, m] @ c[m]) / np.maximum(counts[:, m].sum(1), 1) for g, m in masks.items() if m.sum() >= min_n}
    if not accs:
        return {}, None, None
    S = np.stack(list(accs.values()), 1)
    return {g: ci(a) for g, a in accs.items()}, ci(S.min(1)), ci(S.max(1) - S.min(1))

def boot_pair_auc(dist, y, i, j, counts):
    """Image-level bootstrap of ROC-AUC: pair weight = draws of image i * draws of image j."""
    out = []
    for c in counts:
        w = np.where(i == j, c[i], c[i] * c[j])
        if w[y == 1].sum() == 0 or w[y == 0].sum() == 0:
            continue
        out.append(roc_auc_score(y, -dist, sample_weight=w))
    return np.array(out)

def bootstrap_block(te, correct, counts, min_n=30, race_names=None):
    r_names = list(race_names) if race_names is not None else RACE_NAMES
    r = te.race.values
    masks = {r_names[k]: (r == k) for k in np.unique(r)}
    out = {}
    for m, c in correct.items():
        per, worst, gap = boot_group(c, masks, counts, min_n)
        out[m] = {"acc": float(c.mean()), "acc_ci": ci(boot_accuracy(c, counts)),
                  "by_race_ci": per, "worst_race_ci": worst, "gap_ci": gap}
    return out

def balanced_fairness(te, correct, R=50, min_cell=30, seed=0, race_names=None):
    """Repeatedly subsample every race x gender cell to the smallest cell; mean accuracy per race."""
    r_names = list(race_names) if race_names is not None else RACE_NAMES
    rng = np.random.default_rng(seed)
    r, g = te.race.values, te.gender.values
    cells = {(a, b): np.flatnonzero((r == a) & (g == b)) for a in np.unique(r) for b in np.unique(g)}
    cells = {k: v for k, v in cells.items() if len(v) >= min_cell}
    if not cells:
        return None
    n_per = min(len(v) for v in cells.values())
    races = sorted({k[0] for k in cells})
    acc = {m: np.zeros((R, len(races))) for m in correct}
    for t in range(R):
        draw = {k: rng.choice(v, n_per, replace=False) for k, v in cells.items()}
        for m, c in correct.items():
            for ri, race in enumerate(races):
                acc[m][t, ri] = c[np.concatenate([v for k, v in draw.items() if k[0] == race])].mean()
    return {"n_per_cell": int(n_per), "cells_used": len(cells),
            "metrics": {m: {"by_race": {r_names[race]: float(acc[m][:, ri].mean()) for ri, race in enumerate(races)},
                            "worst": float(acc[m].min(1).mean()),
                            "gap": float((acc[m].max(1) - acc[m].min(1)).mean())} for m in correct}}

def probe_predict(E_tr, lab_tr, E_te, n_race=5, n_age=9, max_train=10000, seed=0):
    """Logistic-regression probes on a frozen embedding -> class probabilities."""
    rng = np.random.default_rng(seed)
    sub = rng.choice(len(E_tr), min(len(E_tr), max_train), replace=False)
    out = {}
    for name, col, k in (("gender", 0, 2), ("race", 1, n_race), ("age_bin", 2, n_age)):
        clf = LogisticRegression(max_iter=5000, tol=1e-3, C=1.0, random_state=seed)
        clf.fit(E_tr[sub], lab_tr[sub, col])
        P = np.zeros((len(E_te), k))
        P[:, clf.classes_] = clf.predict_proba(E_te)
        out[name] = P
    return out

def dataset_probe(E_tr, ds_tr, E_te, ds_te, max_train=10000, seed=0):
    """Can the embedding tell the datasets apart? (shortcut check; None with a single dataset)"""
    if len(set(ds_tr)) < 2 or len(set(ds_te)) < 2:
        return None
    rng = np.random.default_rng(seed)
    sub = rng.choice(len(E_tr), min(len(E_tr), max_train), replace=False)
    clf = LogisticRegression(max_iter=5000, tol=1e-3, C=1.0, random_state=seed)
    clf.fit(E_tr[sub], np.asarray(ds_tr)[sub])
    shares = pd.Series(ds_te).value_counts(normalize=True)
    return {"accuracy": float(clf.score(E_te, ds_te)), "majority_baseline": float(shares.max())}

def attr_block(te, p_gender, p_race, p_age, race_names=None):
    """Metrics + fairness from class probabilities. Returns (block, per-image correctness, predictions)."""
    r_names = list(race_names) if race_names is not None else RACE_NAMES
    g, r, a = te.gender.values, te.race.values, te.age_bin.values
    pg, pr, pa = p_gender.argmax(1), p_race.argmax(1), p_age.argmax(1)
    mae, mae_n = None, 0
    if "age" in te.columns and te["age"].notna().any():
        m = te["age"].notna().values
        mae = float(np.abs((p_age @ AGE_BIN_MIDPOINTS)[m] - te["age"].values[m]).mean()); mae_n = int(m.sum())
    n_age = p_age.shape[1]
    age_cls = cls_report(a, pa, n_age, AGE_BIN_NAMES, probs=p_age)
    top2 = float(top_k_accuracy_score(a, p_age, k=min(2, n_age), labels=list(range(n_age)))) if len(np.unique(a)) > 1 else age_cls["accuracy"]
    top3 = float(top_k_accuracy_score(a, p_age, k=min(3, n_age), labels=list(range(n_age)))) if len(np.unique(a)) > 1 else age_cls["accuracy"]
    qwk = float(cohen_kappa_score(a, pa, weights="quadratic")) if len(np.unique(a)) > 1 else 0.0
    age_blk = {
        **age_cls,
        "bin_accuracy": age_cls["accuracy"],
        "bin_accuracy_pm1": float((np.abs(pa - a) <= 1).mean()),
        "top_2_accuracy": top2,
        "top_3_accuracy": top3,
        "cohen_kappa_quadratic": qwk,
        "mae_midpoint": mae,
        "mae_n": mae_n,
        "bin_names": AGE_BIN_NAMES,
        "bin_confusion_matrix": age_cls["confusion_matrix"],
    }
    blk = {
        "gender": cls_report(g, pg, 2, GENDER_NAMES, probs=p_gender),
        "race": cls_report(r, pr, p_race.shape[1], r_names, probs=p_race),
        "age": age_blk,
        "fairness": fairness_report(te, pg, pr, pa, race_names=r_names),
    }
    return blk, {"gender": pg == g, "race": pr == r, "age_bin": pa == a}, (pg, pr, pa)

def metrics_summary_table(res):
    """Build a unified DataFrame of Accuracy, Precision, Recall, F1, Balanced Acc, ROC-AUC, PR-AUC & MCC across all tasks."""
    va, vi = res["verification"], res["verification_instance"]
    g, r, a = res["gender"], res["race"], res["age"]
    n_race = len(r["class_names"])
    n_age = len(a["class_names"])
    rows = [
        {"Task": "Attr Verification", "Accuracy": va["accuracy"], "Balanced Acc": va["balanced_accuracy"],
         "Precision": va["precision"], "Recall": va["recall"], "F1": va["f1"],
         "F1 (Weighted)": va["f1"], "ROC-AUC": va["roc_auc"], "PR-AUC": va["pr_auc"], "MCC": va["mcc"]},
        {"Task": "Inst Verification", "Accuracy": vi["accuracy"], "Balanced Acc": vi["balanced_accuracy"],
         "Precision": vi["precision"], "Recall": vi["recall"], "F1": vi["f1"],
         "F1 (Weighted)": vi["f1"], "ROC-AUC": vi["roc_auc"], "PR-AUC": vi["pr_auc"], "MCC": vi["mcc"]},
        {"Task": "Gender (2-cls)", "Accuracy": g["accuracy"], "Balanced Acc": g["balanced_accuracy"],
         "Precision": g["precision_macro"], "Recall": g["recall_macro"], "F1": g["f1_macro"],
         "F1 (Weighted)": g["f1_weighted"], "ROC-AUC": g.get("roc_auc_ovr_macro", np.nan),
         "PR-AUC": g.get("pr_auc_macro", np.nan), "MCC": g["mcc"]},
        {"Task": f"Race ({n_race}-cls)", "Accuracy": r["accuracy"], "Balanced Acc": r["balanced_accuracy"],
         "Precision": r["precision_macro"], "Recall": r["recall_macro"], "F1": r["f1_macro"],
         "F1 (Weighted)": r["f1_weighted"], "ROC-AUC": r.get("roc_auc_ovr_macro", np.nan),
         "PR-AUC": r.get("pr_auc_macro", np.nan), "MCC": r["mcc"]},
        {"Task": f"Age Bin ({n_age}-cls)", "Accuracy": a["bin_accuracy"], "Balanced Acc": a["balanced_accuracy"],
         "Precision": a["precision_macro"], "Recall": a["recall_macro"], "F1": a["f1_macro"],
         "F1 (Weighted)": a["f1_weighted"], "ROC-AUC": a.get("roc_auc_ovr_macro", np.nan),
         "PR-AUC": a.get("pr_auc_macro", np.nan), "MCC": a["mcc"]},
    ]
    return pd.DataFrame(rows).set_index("Task")

def per_class_metrics_df(res, task="race"):
    """Return a clean per-class DataFrame (precision, recall, f1, support) for 'gender', 'race', or 'age'."""
    return pd.DataFrame(res[task]["per_class"]).T

**Validation - Step 9** (mini fit, checkpoint round trip, resume, and statistical / evaluation helpers)

In [ ]:
# 1. Training loop, checkpointing, objective weights & resume
mini = dataclasses.replace(cfg, epochs=2, steps_per_epoch=10, n_eval_pairs=200, batch_inst=32,
                           out_dir=str(Path(cfg.out_dir) / "_mini"))
set_seed(mini.seed)
va_cache, va_lab = to_device(build_cache(data["val"].path, cfg.img_size, cfg.cache_dir), data["val"])
va_pairs = PairSampler(data["val"], cfg.group_power, n_race=cfg.n_race, n_age=cfg.n_age).make_fixed(mini.n_eval_pairs, mini.eval_seed + 1)
m = SiameseCNN(mini).to(DEVICE)
hist = fit(m, tr_cache, tr_lab, train_sampler, va_cache, va_lab, va_pairs, mini, mini.out_dir)
check("fit runs and logs one record per epoch", len(hist) == 2)
check("all logged values are finite", all(math.isfinite(v) for h in hist for v in h.values()))
check("last.pt and best.pt written", (Path(mini.out_dir) / "last.pt").exists() and (Path(mini.out_dir) / "best.pt").exists())

tot_c, parts_c, _, _ = evaluate(m, va_cache, va_lab, va_pairs, mini)
only_heads = dataclasses.replace(mini, w_pair=0.0, w_inst=0.0)
tot_a, parts_a, _, _ = evaluate(m, va_cache, va_lab, va_pairs, only_heads)
check("objective with pair losses off equals the head loss (config A)", abs(tot_a - parts_a["heads"]) < 1e-6, f"{tot_a:.4f}")
check("objective is deterministic (fixed pairs, seeded augmentation)",
      abs(evaluate(m, va_cache, va_lab, va_pairs, mini)[0] - tot_c) < 1e-6)

# checkpoint round trip
sd = torch.load(Path(mini.out_dir) / "best.pt", map_location=DEVICE)["model"]
m.load_state_dict(sd); m2 = SiameseCNN(mini).to(DEVICE); m2.load_state_dict(sd)
m.eval(); m2.eval()
with torch.no_grad():
    xx = prep(va_cache[:32], False)
    check("checkpoint round trip reproduces both embeddings",
          all(torch.allclose(a_, b_, atol=1e-5) for a_, b_ in zip(m(xx)[:2], m2(xx)[:2])))

# resume: simulate a disconnect after epoch 1 of a 2-epoch run, then resume with the SAME config
rs = dataclasses.replace(mini, resume=False, out_dir=str(Path(cfg.out_dir) / "_resume"))
shutil.rmtree(rs.out_dir, ignore_errors=True)
set_seed(rs.seed)
hA = fit(SiameseCNN(rs).to(DEVICE), tr_cache, tr_lab, train_sampler, va_cache, va_lab, va_pairs, rs, rs.out_dir, stop_after=1)
check("interrupted run stopped after 1 epoch", len(hA) == 1)
rs2 = dataclasses.replace(rs, resume=True)
hB = fit(SiameseCNN(rs2).to(DEVICE), tr_cache, tr_lab, train_sampler, va_cache, va_lab, va_pairs, rs2, rs2.out_dir)
check("resume continues from the saved epoch and finishes", len(hB) == 2 and hB[-1]["epoch"] == 2)

In [ ]:
# 2. Verification, bootstrap CIs, balanced fairness, probes & instance evaluation checks
rng_ = np.random.default_rng(0)
y = np.tile([1, 0], 500); d = np.where(y == 1, rng_.normal(0.4, 0.1, 1000), rng_.normal(1.0, 0.1, 1000))
vr = verification_report(d, y, np.where(y == 0, 1, 0), best_threshold(d, y))
check("verification_report: separable data -> AUC near 1, PR-AUC near 1, EER near 0",
      vr["roc_auc"] > 0.99 and vr["pr_auc"] > 0.99 and vr["eer"] < 0.05,
      f"AUC {vr['roc_auc']:.3f} PR-AUC {vr['pr_auc']:.3f} EER {vr['eer']:.3f}")
check("verification_report: specificity, npv, balanced_accuracy, mcc in valid range",
      0.0 <= vr["specificity"] <= 1.0 and 0.0 <= vr["npv"] <= 1.0
      and 0.0 <= vr["balanced_accuracy"] <= 1.0 and -1.0 <= vr["mcc"] <= 1.0)
rand = verification_report(rng_.random(1000), y, np.where(y == 0, 1, 0), 0.5)
check("verification_report: random scores -> AUC near 0.5", 0.4 < rand["roc_auc"] < 0.6, f"{rand['roc_auc']:.3f}")

n = 3000
counts = boot_counts(n, 400, 1)
check("bootstrap counts: every replicate draws n images", bool((counts.sum(1) == n).all()))
corr = rng_.random(n) < 0.7
lo, hi = ci(boot_accuracy(corr, counts))
check("accuracy CI contains the estimate and has expected width",
      lo <= corr.mean() <= hi and 0.8 * 2 * 1.96 * math.sqrt(0.21 / n) < hi - lo < 1.3 * 2 * 1.96 * math.sqrt(0.21 / n), f"[{lo:.3f}, {hi:.3f}]")

te_s = pd.DataFrame({"race": np.repeat([0, 1], n // 2), "gender": np.tile([0, 1], n // 2)})
c_s = np.concatenate([rng_.random(n // 2) < 0.9, rng_.random(n // 2) < 0.5])
bb = bootstrap_block(te_s, {"gender": c_s}, counts, race_names=RACE_NAMES)["gender"]
check("worst-group CI tracks the weak group (~0.5)", 0.45 < bb["worst_race_ci"][0] and bb["worst_race_ci"][1] < 0.56, bb["worst_race_ci"])
check("gap CI excludes 0 for a real gap (~0.4)", bb["gap_ci"][0] > 0.3, bb["gap_ci"])

N_img, n_pairs = 400, 3000
i_ = rng_.integers(0, N_img, n_pairs); j_ = rng_.integers(0, N_img, n_pairs)
img_effect = rng_.normal(0, 1, N_img)
y_ = (rng_.random(n_pairs) < 0.5).astype(int)
dist_ = -(0.8 * y_ + 0.5 * (img_effect[i_] + img_effect[j_])) + rng_.normal(0, 0.3, n_pairs)
cnt = boot_counts(N_img, 300, 2)
a_img = boot_pair_auc(dist_, y_, i_, j_, cnt)
a_pair = np.array([roc_auc_score(y_[b], -dist_[b]) for b in (rng_.integers(0, n_pairs, n_pairs) for _ in range(300))])
w_img, w_pair = np.subtract(*ci(a_img)[::-1]), np.subtract(*ci(a_pair)[::-1])
check("image-level AUC CI is wider than naive pair-level CI when images are shared", w_img > w_pair, f"{w_img:.3f} vs {w_pair:.3f}")
check("image-level AUC CI contains the point estimate", ci(a_img)[0] <= roc_auc_score(y_, -dist_) <= ci(a_img)[1])

te_b = pd.DataFrame({"race": rng_.integers(0, 3, 4000), "gender": rng_.integers(0, 2, 4000)})
te_b = te_b[~((te_b.race == 2) & (te_b.gender == 1) & (np.arange(len(te_b)) % 4 != 0))].reset_index(drop=True)
cb = np.zeros(len(te_b), bool)
for (rc, gd), idx_ in te_b.groupby(["race", "gender"]).indices.items():
    cb[idx_[: int(round((0.9 if gd == 0 else 0.5) * len(idx_)))]] = True
plain = [cb[te_b.race.values == k].mean() for k in range(3)]
bf = balanced_fairness(te_b, {"m": cb}, R=30, race_names=RACE_NAMES)
vals = list(bf["metrics"]["m"]["by_race"].values())
check("plain race accuracy is biased by the thinned cell (> 5 points apart)", max(plain) - min(plain) > 0.05, np.round(plain, 3))
check("balanced fairness removes group-size effect (within 2 points of 0.70)",
      max(vals) - min(vals) < 0.02 and abs(np.mean(vals) - 0.70) < 0.02, np.round(vals, 3))

lab = np.stack([rng_.integers(0, 2, 2000), rng_.integers(0, 5, 2000), rng_.integers(0, 9, 2000)], 1)
E = rng_.normal(0, 1, (2000, 16)); E[:, 0] += 6 * lab[:, 0]; E[:, 1:3] += 6 * np.eye(5)[lab[:, 1]][:, :2]
Pp = probe_predict(E[:1500], lab[:1500], E[1500:])
check("probe: gender recoverable from a signal embedding (> 0.95)", (Pp["gender"].argmax(1) == lab[1500:, 0]).mean() > 0.95)
Pn = probe_predict(rng_.normal(0, 1, (1500, 16)), lab[:1500], rng_.normal(0, 1, (500, 16)))
check("probe: noise embedding gives chance-level gender accuracy (< 0.6)", (Pn["gender"].argmax(1) == lab[1500:, 0]).mean() < 0.6)
check("probe outputs are probability rows", all(np.allclose(P.sum(1), 1, atol=1e-5) for P in Pp.values()))
check("dataset probe returns None with a single dataset", dataset_probe(E[:100], ["a"] * 100, E[:10], ["a"] * 10) is None)

te_a = pd.DataFrame({"gender": lab[:, 0], "race": lab[:, 1], "age_bin": lab[:, 2], "age": rng_.integers(1, 90, 2000).astype(float)})
blk, _, _ = attr_block(te_a, np.eye(2)[lab[:, 0]], np.eye(5)[lab[:, 1]], np.eye(9)[lab[:, 2]], race_names=RACE_NAMES)
check("attr_block: perfect probabilities -> accuracy, precision, recall, F1 = 1 everywhere",
      blk["gender"]["accuracy"] == 1 and blk["race"]["precision_macro"] == 1
      and blk["race"]["recall_macro"] == 1 and blk["age"]["f1_macro"] == 1
      and blk["age"]["balanced_accuracy"] == 1)
check("attr_block: per-class support sums to n",
      sum(v["support"] for v in blk["race"]["per_class"].values()) == 2000
      and sum(v["support"] for v in blk["age"]["per_class"].values()) == 2000)
check("attr_block: confusion matrices sum to n", np.sum(blk["gender"]["confusion_matrix"]) == 2000 and np.sum(blk["age"]["bin_confusion_matrix"]) == 2000)

set_seed(cfg.seed)
mm = SiameseCNN(cfg).to(DEVICE)
I1 = instance_eval(mm, va_cache, cfg, seed=5)
I2 = instance_eval(mm, va_cache, cfg, seed=5)
n_i = len(I1["y"]) // 2
check("instance eval: n positives == n negatives", I1["y"].sum() == n_i)
check("instance eval: positives pair image with itself, negatives with different image",
      bool((I1["i"][:n_i] == I1["j"][:n_i]).all()) and bool((I1["i"][n_i:] != I1["j"][n_i:]).all()))
check("instance eval is deterministic for a fixed seed", np.allclose(I1["dist"], I2["dist"], atol=1e-5))
check("distances are within unit-sphere range [0, 2]", bool((I1["dist"] >= 0).all() and (I1["dist"] <= 2.001).all()))
Pq = predict_all(mm, va_cache[:100], cfg)
check("predict_all: probability rows sum to 1", np.allclose(Pq["p_gender"].sum(1), 1, atol=1e-4) and np.allclose(Pq["p_race"].sum(1), 1, atol=1e-4)
      and np.allclose(Pq["p_age"].sum(1), 1, atol=1e-4))
check("predict_all: embeddings are unit length", np.allclose(np.linalg.norm(Pq["emb_attr"], axis=1), 1, atol=1e-3)
      and np.allclose(np.linalg.norm(Pq["emb_inst"], axis=1), 1, atol=1e-3))
del mm

## Step 10 - `run_siamese_cnn(data, config)`: core training & evaluation engine

Takes the `train`/`val`/`test` DataFrames and a `Config`, trains `SiameseCNN`, selects verification thresholds strictly on validation, evaluates on test, computes bootstrap CIs and fairness metrics, saves `siamese_best.pt` and `results.json`, and returns the full metrics dictionary.

In [ ]:
def make_summary(res):
    f = lambda c: f"[{c[0]:.3f}, {c[1]:.3f}]" if c else "n/a"
    v, vi, g, r, a = res["verification"], res["verification_instance"], res["gender"], res["race"], res["age"]
    P = res["primary_source"]; B = res["bootstrap"][P]
    tbl = metrics_summary_table(res).round(3).to_string()
    L = [f"=== Siamese CNN ({res['config']['name']}, seed {res['config']['seed']}) ===",
         f"Params {res['n_params']:,} | train {res['train_time_s']:.0f}s | best epoch {res['best_epoch']} | attribute readout: {P}",
         f"Attribute-pair verification: ROC-AUC {v['roc_auc']:.3f} {f(res['bootstrap_verification']['attr_auc_ci'])} | PR-AUC {v['pr_auc']:.3f} | EER {v['eer']:.3f} | "
         f"Acc {v['accuracy']:.3f} | Prec {v['precision']:.3f} | Rec {v['recall']:.3f} | F1 {v['f1']:.3f} | hard-neg AUC {v['auc_by_ndiff'].get(1, float('nan')):.3f}",
         f"Instance verification:       ROC-AUC {vi['roc_auc']:.3f} {f(res['bootstrap_verification']['inst_auc_ci'])} | PR-AUC {vi['pr_auc']:.3f} | EER {vi['eer']:.3f} | "
         f"Acc {vi['accuracy']:.3f} | Prec {vi['precision']:.3f} | Rec {vi['recall']:.3f} | F1 {vi['f1']:.3f}",
         f"Gender: acc {g['accuracy']:.3f} {f(B['gender']['acc_ci'])} | bal-acc {g['balanced_accuracy']:.3f} | "
         f"prec {g['precision_macro']:.3f} | rec {g['recall_macro']:.3f} | macro-F1 {g['f1_macro']:.3f} | AUC {g.get('roc_auc_ovr_macro', float('nan')):.3f}",
         f"Race:   acc {r['accuracy']:.3f} {f(B['race']['acc_ci'])} | bal-acc {r['balanced_accuracy']:.3f} | "
         f"prec {r['precision_macro']:.3f} | rec {r['recall_macro']:.3f} | macro-F1 {r['f1_macro']:.3f} | w-F1 {r['f1_weighted']:.3f} | OvR-AUC {r.get('roc_auc_ovr_macro', float('nan')):.3f}",
         f"Age:    bin acc {a['bin_accuracy']:.3f} {f(B['age_bin']['acc_ci'])} | +-1 bin {a['bin_accuracy_pm1']:.3f} | bal-acc {a['balanced_accuracy']:.3f} | "
         f"prec {a['precision_macro']:.3f} | rec {a['recall_macro']:.3f} | macro-F1 {a['f1_macro']:.3f} | w-F1 {a['f1_weighted']:.3f}" +
         (f" | midpoint MAE {a['mae_midpoint']:.2f} yrs (n={a['mae_n']})" if a["mae_midpoint"] is not None else ""),
         "",
         "--- Unified Classification & Verification Metrics ---",
         tbl,
         "",
         "--- Race Per-Class Metrics ---",
         per_class_metrics_df(res, "race").round(3).to_string(),
         "",
         "Race groups (natural test distribution): worst-group accuracy [95% CI] / gap [95% CI]"]
    for m in ("gender", "race", "age_bin"):
        b = B[m]
        L.append(f"  {m:<8} worst {f(b['worst_race_ci'])} | gap {f(b['gap_ci'])}")
    if res["fairness_balanced"] and P in res["fairness_balanced"] and res["fairness_balanced"][P]:
        bal = res["fairness_balanced"][P]
        L.append(f"Balanced test (n={bal['n_per_cell']} per race x gender cell): " +
                 " | ".join(f"{m} worst {x['worst']:.3f} gap {x['gap']:.3f}" for m, x in bal["metrics"].items()))
    if res["dataset_probe"]:
        L.append(f"Dataset shortcut probe: acc {res['dataset_probe']['accuracy']:.3f} (majority baseline {res['dataset_probe']['majority_baseline']:.3f})")
    return "\n".join(L)

def run_siamese_cnn(data, config=None, verbose=True):
    cfg = sync_config_schema(dataclasses.replace(config) if config is not None else make_config())
    cfg.amp = cfg.amp and DEVICE.type == "cuda"
    r_names = list(cfg.race_names)
    need = {"path", "gender", "race", "age_bin"}
    for k in ("train", "val", "test"):
        if k not in data or not need <= set(data[k].columns):
            raise ValueError(f"data['{k}'] must be a DataFrame with columns {sorted(need)} (+ optional 'age', 'dataset')")
    set_seed(cfg.seed)
    run_tag = f"{cfg.dataset.lower()}_{cfg.race_schema}_s{cfg.seed}"
    base_out = Path(cfg.out_dir)
    out = base_out if run_tag in str(base_out) else base_out / run_tag
    out.mkdir(parents=True, exist_ok=True)
    cfg.out_dir = str(out)
    tr, va, te = (data[k].reset_index(drop=True) for k in ("train", "val", "test"))
    tr_cache, tr_lab = to_device(build_cache(tr.path, cfg.img_size, cfg.cache_dir), tr)
    va_cache, va_lab = to_device(build_cache(va.path, cfg.img_size, cfg.cache_dir), va)
    te_cache, te_lab = to_device(build_cache(te.path, cfg.img_size, cfg.cache_dir), te)
    tr_sampler = PairSampler(tr, cfg.group_power, n_race=cfg.n_race, n_age=cfg.n_age)
    va_pairs = PairSampler(va, cfg.group_power, n_race=cfg.n_race, n_age=cfg.n_age).make_fixed(cfg.n_eval_pairs, cfg.eval_seed + 1)
    te_pairs = PairSampler(te, cfg.group_power, n_race=cfg.n_race, n_age=cfg.n_age).make_fixed(cfg.n_eval_pairs, cfg.eval_seed + 2)

    model = SiameseCNN(cfg).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    t0 = time.time()
    history = fit(model, tr_cache, tr_lab, tr_sampler, va_cache, va_lab, va_pairs, cfg, out, verbose=verbose)
    train_time = time.time() - t0
    best = torch.load(out / "best.pt", map_location=DEVICE)
    model.load_state_dict(best["model"])

    # thresholds chosen on validation only
    _, _, vd, vy = evaluate(model, va_cache, va_lab, va_pairs, cfg)
    thr_attr = best_threshold(vd, vy)
    vi = instance_eval(model, va_cache, cfg, cfg.eval_seed + 3)
    thr_inst = best_threshold(vi["dist"], vi["y"])

    # test evaluation
    _, _, td, ty = evaluate(model, te_cache, te_lab, te_pairs, cfg)
    ti = instance_eval(model, te_cache, cfg, cfg.eval_seed + 4)
    ver_attr = verification_report(td, ty, te_pairs["ndiff"], thr_attr)
    ver_inst = verification_report(ti["dist"], ti["y"], ti["ndiff"], thr_inst)

    P_te, P_tr = predict_all(model, te_cache, cfg), predict_all(model, tr_cache, cfg)
    heads_trained = cfg.attr_weight > 0
    blocks, corrects, preds = {}, {}, {}
    if heads_trained:
        blocks["heads"], corrects["heads"], preds["heads"] = attr_block(te, P_te["p_gender"], P_te["p_race"], P_te["p_age"], race_names=r_names)
    pr_ = probe_predict(P_tr["emb_attr"], tr[["gender", "race", "age_bin"]].values, P_te["emb_attr"], cfg.n_race, cfg.n_age, seed=cfg.seed)
    blocks["probe"], corrects["probe"], preds["probe"] = attr_block(te, pr_["gender"], pr_["race"], pr_["age_bin"], race_names=r_names)
    primary = "heads" if heads_trained else "probe"

    counts = boot_counts(len(te), cfg.n_boot, cfg.eval_seed)
    boot = {src: bootstrap_block(te, corrects[src], counts, race_names=r_names) for src in corrects}
    boot_ver = {"attr_auc_ci": ci(boot_pair_auc(td, ty, te_pairs["i"], te_pairs["j"], counts)),
                "inst_auc_ci": ci(boot_pair_auc(ti["dist"], ti["y"], ti["i"], ti["j"], counts))}
    bal = {src: balanced_fairness(te, corrects[src], seed=cfg.eval_seed, race_names=r_names) for src in corrects}
    ds_probe = dataset_probe(P_tr["emb_attr"], tr["dataset"].values, P_te["emb_attr"], te["dataset"].values, seed=cfg.seed)         if "dataset" in tr.columns else None

    res = {"model": "siamese_cnn", "config": asdict(cfg), "n_params": n_params, "train_time_s": train_time,
           "best_epoch": int(best["epoch"]), "primary_source": primary,
           "stopped_epoch": int(history[-1]["epoch"]),
           "early_stopped": len(history) < cfg.epochs,
           "history": history,
           "verification": ver_attr, "verification_instance": ver_inst,
           "bootstrap_verification": boot_ver, "bootstrap": boot, "fairness_balanced": bal,
           "heads": blocks.get("heads"), "probe": blocks["probe"], "dataset_probe": ds_probe,
           "gender": blocks[primary]["gender"], "race": blocks[primary]["race"], "age": blocks[primary]["age"],
           "fairness": blocks[primary]["fairness"],
           "predictions": {"path": te.path.tolist(), "gender": preds[primary][0].tolist(),
                           "race": preds[primary][1].tolist(), "age_bin": preds[primary][2].tolist()}}
    res["metrics_table"] = metrics_summary_table(res).reset_index().to_dict(orient="records")
    res["checkpoint"] = str(out / "siamese_best.pt")
    torch.save({"model": best["model"], "config": asdict(cfg), "thresholds": {"attr": thr_attr, "inst": thr_inst},
                "n_race": cfg.n_race, "n_age": cfg.n_age, "age_edges": AGE_EDGES,
                "age_bin_names": AGE_BIN_NAMES, "race_names": r_names,
                "primary_source": primary}, res["checkpoint"])
    np.save(out / "test_embeddings_attr.npy", P_te["emb_attr"])
    np.save(out / "test_embeddings_inst.npy", P_te["emb_inst"])
    res["summary"] = make_summary(res)
    (out / "results.json").write_text(json.dumps(res, indent=2, default=_json_default))
    if verbose:
        print(res["summary"])
    return res

## Step 11 - Run full model (`C_full`), validate results, and plot diagnostics

Runs `C_full` through `run_siamese_cnn`, validates all output structures, bootstrap CIs, and probe fallback behavior, and renders diagnostic charts via `plot_diagnostics(results)`.

In [ ]:
def plot_cm(ax, cm, names, title):
    cm = np.array(cm); norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(norm, vmin=0, vmax=1, cmap="Blues"); ax.set_title(title)
    ax.set_xticks(range(len(names))); ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8); ax.set_yticklabels(names, fontsize=8)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if norm[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted"); ax.set_ylabel("true")

def plot_diagnostics(results: Dict[str, Any]):
    r_names = list(results["race"]["class_names"])
    H = pd.DataFrame(results["history"])
    fig, ax = plt.subplots(1, 4, figsize=(20, 4))
    ax[0].plot(H.epoch, H.train_loss, marker="o", label="train"); ax[0].plot(H.epoch, H.val_loss, marker="o", label="val")
    ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()

    ax[1].plot(H.epoch, H.val_attr_f1, marker="o", label="attr-pair F1")
    ax[1].plot(H.epoch, H.val_gender_f1, marker="o", label="gender F1")
    ax[1].plot(H.epoch, H.val_race_f1, marker="o", label="race F1")
    ax[1].plot(H.epoch, H.val_age_f1, marker="o", label="age F1")
    ax[1].set_title("validation macro-F1 by epoch"); ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1); ax[1].legend(fontsize=8)

    roc_a = results["verification"]["roc"]
    roc_i = results["verification_instance"]["roc"]
    ax[2].plot(roc_a["fpr"], roc_a["tpr"], label=f"attr (AUC {results['verification']['roc_auc']:.3f})")
    ax[2].plot(roc_i["fpr"], roc_i["tpr"], label=f"inst (AUC {results['verification_instance']['roc_auc']:.3f})")
    ax[2].plot([0, 1], [0, 1], "k--", lw=0.8)
    ax[2].set_title("test ROC curves"); ax[2].set_xlabel("FPR"); ax[2].set_ylabel("TPR"); ax[2].legend(fontsize=8)

    pr_a = results["verification"]["pr_curve"]
    pr_i = results["verification_instance"]["pr_curve"]
    ax[3].plot(pr_a["recall"], pr_a["precision"], label=f"attr (PR-AUC {results['verification']['pr_auc']:.3f})")
    ax[3].plot(pr_i["recall"], pr_i["precision"], label=f"inst (PR-AUC {results['verification_instance']['pr_auc']:.3f})")
    ax[3].axhline(0.5, color="k", linestyle="--", lw=0.8)
    ax[3].set_title("test Precision-Recall curves"); ax[3].set_xlabel("Recall"); ax[3].set_ylabel("Precision"); ax[3].legend(fontsize=8)
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots(1, 3, figsize=(17, 5))
    plot_cm(ax[0], results["gender"]["confusion_matrix"], GENDER_NAMES, "gender")
    plot_cm(ax[1], results["race"]["confusion_matrix"], r_names, "race")
    plot_cm(ax[2], results["age"]["bin_confusion_matrix"], AGE_BIN_NAMES, "age bin")
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(16, 4))
    for idx_t, (task_nm, title) in enumerate((("race", "Race: per-class Precision / Recall / F1"),
                                              ("age", "Age Bin: per-class Precision / Recall / F1"))):
        df_pc = per_class_metrics_df(results, task_nm)[["precision", "recall", "f1"]]
        df_pc.plot.bar(ax=ax[idx_t], ylim=(0, 1), rot=30)
        ax[idx_t].set_title(title); ax[idx_t].legend(loc="lower right", fontsize=8)
    plt.tight_layout(); plt.show()

    fr = pd.DataFrame(results["fairness"]["by_race"]).T
    fr[["gender_acc", "race_acc", "age_bin_acc"]].plot.bar(figsize=(8, 3.5), title="accuracy by race group (test)")
    plt.ylim(0, 1); plt.show()

cfg_run = dataclasses.replace(cfg, name="C_full", out_dir=str(Path(cfg.out_dir) / "C_full"))
results = run_siamese_cnn(data, cfg_run)

**Validation - Step 11**

In [ ]:
def validate_results_block(R: Dict[str, Any], data: Dict[str, pd.DataFrame], cfg_ref: Config,
                           tracker: Optional[CheckTracker] = None, check_config_b: bool = True):
    keys = {"model", "config", "n_params", "train_time_s", "best_epoch", "history", "verification", "verification_instance",
            "bootstrap", "bootstrap_verification", "fairness_balanced", "heads", "probe", "gender", "race", "age",
            "fairness", "predictions", "metrics_table", "checkpoint", "summary", "primary_source"}
    check("results dict has every expected key", keys <= set(R), keys - set(R), tracker=tracker)
    unit = lambda v: 0.0 <= v <= 1.0
    for nm in ("verification", "verification_instance"):
        check(f"{nm}: metrics in [0, 1]", all(unit(R[nm][k]) for k in ("roc_auc", "pr_auc", "eer", "accuracy", "balanced_accuracy", "precision", "recall", "specificity", "npv", "f1")), tracker=tracker)
        check(f"{nm}: MCC in [-1, 1]", -1.0 <= R[nm]["mcc"] <= 1.0, tracker=tracker)
    check("attribute verification has 50% positives, instance verification too",
          abs(R["verification"]["positive_rate"] - 0.5) < 0.01 and abs(R["verification_instance"]["positive_rate"] - 0.5) < 0.01,
          tracker=tracker)
    check("gender / race / age accuracies in [0, 1]",
          all(unit(x) for x in (R["gender"]["accuracy"], R["race"]["accuracy"], R["age"]["bin_accuracy"], R["age"]["bin_accuracy_pm1"])),
          tracker=tracker)
    check("+-1 bin accuracy >= exact bin accuracy", R["age"]["bin_accuracy_pm1"] >= R["age"]["bin_accuracy"], tracker=tracker)
    cls_metric_keys = ("accuracy", "balanced_accuracy", "precision_macro", "recall_macro", "f1_macro", "precision_weighted", "recall_weighted", "f1_weighted")
    for task_nm in ("gender", "race", "age"):
        check(f"{task_nm}: classification metrics (accuracy, precision, recall, F1, balanced_acc) in [0, 1]",
              all(unit(R[task_nm][k]) for k in cls_metric_keys), tracker=tracker)
        check(f"{task_nm}: per-class precision/recall/F1 present and supports sum to test size",
              sum(v["support"] for v in R[task_nm]["per_class"].values()) == len(data["test"]), tracker=tracker)
    n_test = len(data["test"])
    srcs = [s for s in ("heads", "probe") if R.get(s) is not None]
    check("confusion matrices sum to the test size (active heads/probe)",
          all(np.sum(R[s][k]["confusion_matrix"]) == n_test for s in srcs for k in ("gender", "race", "age"))
          and np.sum(R["age"]["bin_confusion_matrix"]) == n_test, tracker=tracker)
    check("one prediction per test image", len(R["predictions"]["path"]) == n_test, tracker=tracker)
    expected_src = "heads" if R["heads"] is not None else "probe"
    check("primary source matches active heads",
          R["primary_source"] == expected_src and R["gender"] == R[expected_src]["gender"], tracker=tracker)
    ci_ok = lambda c: c[0] <= c[1]
    check("bootstrap intervals are ordered and contain the point estimate",
          all(ci_ok(R["bootstrap"][s][m]["acc_ci"]) and R["bootstrap"][s][m]["acc_ci"][0] <= R["bootstrap"][s][m]["acc"] <= R["bootstrap"][s][m]["acc_ci"][1]
              for s in R["bootstrap"] for m in R["bootstrap"][s]), tracker=tracker)
    bv = R["bootstrap_verification"]
    check("AUC intervals are ordered and bracket the point estimates (+- 0.02 slack)",
          bv["attr_auc_ci"][0] - 0.02 <= R["verification"]["roc_auc"] <= bv["attr_auc_ci"][1] + 0.02
          and bv["inst_auc_ci"][0] - 0.02 <= R["verification_instance"]["roc_auc"] <= bv["inst_auc_ci"][1] + 0.02
          and bv["attr_auc_ci"][0] <= bv["attr_auc_ci"][1] and bv["inst_auc_ci"][0] <= bv["inst_auc_ci"][1],
          tracker=tracker)
    check("balanced fairness computed", R["fairness_balanced"][R["primary_source"]] is not None, tracker=tracker)
    check("history has no NaN", all(math.isfinite(v) for h in R["history"] for v in h.values()), tracker=tracker)
    check("checkpoint, results.json written and valid",
          Path(R["checkpoint"]).exists() and bool(json.loads((Path(R["config"]["out_dir"]) / "results.json").read_text())),
          tracker=tracker)

    if check_config_b:
        cfg_B = dataclasses.replace(cfg_ref, name="B_pairs_only", w_pair=1.0, w_inst=1.0, attr_weight=0.0, epochs=1, steps_per_epoch=10,
                                    n_eval_pairs=200, n_boot=50, out_dir=str(Path(cfg_ref.out_dir) / "B_probe_check"))
        RB = run_siamese_cnn(data, cfg_B, verbose=False)
        check("config B: heads block is None, primary readout is the linear probe", RB["heads"] is None and RB["primary_source"] == "probe", tracker=tracker)
        check("config B still returns the full pipeline keys", {"gender", "race", "age", "fairness", "verification"} <= set(RB), tracker=tracker)

validate_results_block(results, data, cfg)

n_test = len(data["test"])
chance = {"gender": 0.5, "race": float(np.bincount(data["test"].race).max() / n_test)}
for name, val, base in (("verification AUC", results["verification"]["roc_auc"], 0.5),
                        ("instance AUC", results["verification_instance"]["roc_auc"], 0.5),
                        ("gender accuracy", results["gender"]["accuracy"], chance["gender"]),
                        ("race accuracy", results["race"]["accuracy"], chance["race"])):
    print(("OK    " if val > base + 0.03 else "WARN  ") + f"{name} {val:.3f} vs baseline {base:.3f}" +
          ("" if val > base + 0.03 else "  (expected to be low in QUICK_RUN; investigate if it stays here on the full run)"))

In [ ]:
if cfg.show_plots:
    plot_diagnostics(results)
print("\nFull Age-Bin Classification Report:")
print(results["age"]["classification_report"])
fr = pd.DataFrame(results["fairness"]["by_race"]).T
print(fr.round(3))

## Step 12 - A / B / C ablation over seeds

Trains `A_heads_only`, `B_pairs_only`, and `C_full` across multiple seeds with identical data splits and fixed evaluation pairs, logging resumable results to `ablation_results.csv`.

In [ ]:
ABLATIONS = {
    "A_heads_only": dict(w_pair=0.0, w_inst=0.0, attr_weight=1.0),
    "B_pairs_only": dict(w_pair=1.0, w_inst=1.0, attr_weight=0.0),
    "C_full":       dict(w_pair=1.0, w_inst=1.0, attr_weight=1.0),
}
EXTRA_CONFIGS = False
if EXTRA_CONFIGS:
    ABLATIONS["C_attr_pairs_only"] = dict(w_pair=1.0, w_inst=0.0, attr_weight=1.0)
    ABLATIONS["C_inst_pairs_only"] = dict(w_pair=0.0, w_inst=1.0, attr_weight=1.0)

ABLATION_SEEDS = cfg.ablation_seeds
ABLATION_FULL_OVERRIDES = {}
ABLATION_QUICK = dict(epochs=1, steps_per_epoch=10, n_eval_pairs=200, n_boot=50, batch_inst=32)
ABLATION_DIR = Path(cfg.out_dir) / "ablation"

def _g(d, *path, default=np.nan):
    for p in path:
        if d is None or p not in d or d[p] is None:
            return default
        d = d[p]
    return d

def flatten_result(res, cfg):
    row = {"name": cfg.name, "seed": cfg.seed, "epochs": cfg.epochs, "train_time_s": res["train_time_s"],
           "attr_auc": res["verification"]["roc_auc"], "attr_pr_auc": res["verification"]["pr_auc"],
           "attr_eer": res["verification"]["eer"], "attr_acc": res["verification"]["accuracy"],
           "attr_prec": res["verification"]["precision"], "attr_rec": res["verification"]["recall"],
           "attr_f1": res["verification"]["f1"],
           "attr_auc_hard": res["verification"]["auc_by_ndiff"].get(1, np.nan),
           "inst_auc": res["verification_instance"]["roc_auc"], "inst_pr_auc": res["verification_instance"]["pr_auc"],
           "inst_eer": res["verification_instance"]["eer"], "inst_f1": res["verification_instance"]["f1"]}
    for src in ("heads", "probe"):
        b = res[src]
        row[f"{src}_gender_acc"] = _g(b, "gender", "accuracy")
        row[f"{src}_gender_prec"] = _g(b, "gender", "precision_macro")
        row[f"{src}_gender_rec"] = _g(b, "gender", "recall_macro")
        row[f"{src}_gender_f1"] = _g(b, "gender", "f1_macro")
        row[f"{src}_race_acc"] = _g(b, "race", "accuracy")
        row[f"{src}_race_bal_acc"] = _g(b, "race", "balanced_accuracy")
        row[f"{src}_race_prec"] = _g(b, "race", "precision_macro")
        row[f"{src}_race_rec"] = _g(b, "race", "recall_macro")
        row[f"{src}_race_f1"] = _g(b, "race", "f1_macro")
        row[f"{src}_agebin_acc"] = _g(b, "age", "bin_accuracy")
        row[f"{src}_agebin_bal_acc"] = _g(b, "age", "balanced_accuracy")
        row[f"{src}_agebin_prec"] = _g(b, "age", "precision_macro")
        row[f"{src}_agebin_rec"] = _g(b, "age", "recall_macro")
        row[f"{src}_agebin_f1"] = _g(b, "age", "f1_macro")
        row[f"{src}_age_pm1"] = _g(b, "age", "bin_accuracy_pm1")
        row[f"{src}_race_worst"] = _g(b, "fairness", "summary", "by_race.race_acc", "worst")
        row[f"{src}_race_gap"] = _g(b, "fairness", "summary", "by_race.race_acc", "gap")
    return row

def run_ablation(data, base_cfg, ablations=None, seeds=None, quick_overrides=None, ablation_dir=None, verbose=True):
    ablations = ablations or ABLATIONS
    seeds = seeds if seeds is not None else ((0, 1) if base_cfg.quick_run else base_cfg.ablation_seeds)
    quick_overrides = quick_overrides if quick_overrides is not None else (ABLATION_QUICK if base_cfg.quick_run else ABLATION_FULL_OVERRIDES)
    ab_dir = Path(ablation_dir) if ablation_dir is not None else (Path(base_cfg.out_dir) / "ablation")
    ab_dir.mkdir(parents=True, exist_ok=True)
    csv = ab_dir / "ablation_results.csv"
    rows = pd.read_csv(csv).to_dict("records") if csv.exists() else []
    done = {(r["name"], int(r["seed"])) for r in rows}
    for name, ov in ablations.items():
        for s in seeds:
            if (name, s) in done:
                if verbose:
                    print(f"skip {name} seed {s} (already in {csv.name})")
                continue
            cfg_i = dataclasses.replace(base_cfg, name=name, seed=s, resume=False, out_dir=str(ab_dir / f"{name}_seed{s}"),
                                        **ov, **(quick_overrides or {}))
            t0 = time.time()
            res = run_siamese_cnn(data, cfg_i, verbose=False)
            rows.append(flatten_result(res, cfg_i))
            pd.DataFrame(rows).to_csv(csv, index=False)
            if verbose:
                print(f"done {name} seed {s} in {time.time()-t0:.0f}s -> {csv.name} ({len(rows)} rows)")
    return pd.DataFrame(rows)

def summarize_ablation(df, cols=None):
    cols = cols or [c for c in df.columns if c not in ("name", "seed", "epochs", "train_time_s")]
    g = df.groupby("name")[cols]
    mean, std = g.mean(), g.std(ddof=1)
    return mean, std, g.size()

def format_ablation_table(df):
    mean, std, n = summarize_ablation(df)
    show = ["attr_auc", "attr_pr_auc", "attr_f1", "inst_auc",
            "heads_gender_f1", "heads_race_acc", "heads_race_prec", "heads_race_rec", "heads_race_f1", "heads_agebin_f1",
            "probe_race_f1", "probe_agebin_f1", "probe_race_worst"]
    show = [c for c in show if c in mean.columns]
    return pd.DataFrame({c: [f"{mean.loc[k, c]:.3f} +- {0 if np.isnan(std.loc[k, c]) else std.loc[k, c]:.3f}" if not np.isnan(mean.loc[k, c]) else "n/a"
                             for k in mean.index] for c in show}, index=[f"{k} (n={n[k]})" for k in mean.index])

def print_ablation(df):
    print(format_ablation_table(df).to_string())

if cfg.run_ablation:
    ablation_df = run_ablation(data, cfg, ABLATIONS, ablation_dir=ABLATION_DIR)
    print_ablation(ablation_df)
else:
    ablation_df = None
    print("SKIP ablation (run_ablation = False)")

**Validation - Step 12** (aggregation against known values, results CSV, and ablation grid)

In [ ]:
if cfg.run_ablation:
    toy = pd.DataFrame({"name": ["X"] * 3 + ["Y"] * 3, "seed": [0, 1, 2] * 2, "epochs": 1, "train_time_s": 1.0,
                        "attr_auc": [0.6, 0.7, 0.8, 0.5, 0.5, 0.5], "heads_race_acc": [np.nan] * 3 + [0.4, 0.5, 0.6]})
    m_, s_, _ = summarize_ablation(toy, ["attr_auc", "heads_race_acc"])
    check("aggregation: mean and std over seeds", abs(m_.loc["X", "attr_auc"] - 0.7) < 1e-9 and abs(s_.loc["X", "attr_auc"] - 0.1) < 1e-9 and abs(m_.loc["Y", "heads_race_acc"] - 0.5) < 1e-9)
    check("aggregation: config without trained heads keeps NaN (not 0)", np.isnan(m_.loc["X", "heads_race_acc"]))
    check("every (config, seed) pair has exactly one row", ablation_df.groupby(["name", "seed"]).size().eq(1).all())
    exp_names = set(ABLATIONS); exp_seeds = {0, 1} if cfg.quick_run else set(cfg.ablation_seeds)
    check("full grid present", {(a, s) for a in exp_names for s in exp_seeds} <= {(r.name, int(r.seed)) for r in ablation_df.itertuples()})
    check("ablation A has non-NaN heads and B has NaN heads (as designed)",
          ablation_df.loc[ablation_df.name == "A_heads_only", "heads_race_acc"].notna().all()
          and ablation_df.loc[ablation_df.name == "B_pairs_only", "heads_race_acc"].isna().all())
    check("probe metrics exist for every config", ablation_df["probe_race_acc"].notna().all())
    check("results CSV on disk matches in-memory table", len(pd.read_csv(ABLATION_DIR / "ablation_results.csv")) == len(ablation_df))
    a_auc = ablation_df.loc[ablation_df.name == "A_heads_only", "attr_auc"].mean()
    c_auc = ablation_df.loc[ablation_df.name == "C_full", "attr_auc"].mean()
    print(f"info: attribute-pair AUC  A {a_auc:.3f} vs C {c_auc:.3f}  (A has no pair loss, so C should be higher once trained)")

    if cfg.show_plots:
        mean, std, _ = summarize_ablation(ablation_df)
        panels = [("attr_auc", "attribute-pair AUC"), ("inst_auc", "instance AUC"), ("heads_race_acc", "race acc (heads)"),
                  ("probe_race_acc", "race acc (probe)"), ("probe_gender_acc", "gender acc (probe)"), ("probe_agebin_acc", "age-bin acc (probe)")]
        fig, axs = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.4))
        for ax, (c, title) in zip(axs, panels):
            ax.bar(range(len(mean)), mean[c].values, yerr=std[c].fillna(0).values, capsize=3)
            ax.set_xticks(range(len(mean))); ax.set_xticklabels([k.split("_")[0] for k in mean.index]); ax.set_title(title, fontsize=9)
        plt.tight_layout(); plt.show()
else:
    print("SKIP (run_ablation = False)")

## Step 13 - Inference driver (`run_siamese_pipeline`)

`run_siamese_pipeline(image, checkpoint, reference=None)` accepts a file path, PIL image, or RGB numpy array, automatically loads `race_names`, `age_bin_names`, `n_race`, and `thresholds` from the checkpoint, and supports optional Viola-Jones face detection and pair comparison.

In [ ]:
class SiameseFaceModel:
    """Loaded checkpoint + preprocessing, matching the training cache resizing."""
    def __init__(self, checkpoint, device=None):
        self.device = torch.device(device) if device else DEVICE
        ck = torch.load(checkpoint, map_location=self.device)
        c = {k: v for k, v in ck["config"].items() if k in Config.__dataclass_fields__}
        self.cfg = sync_config_schema(Config(**c))
        self.thr = ck["thresholds"]
        self.race_names = list(ck["race_names"])
        self.age_names = list(ck["age_bin_names"])
        self.cfg.race_names = tuple(self.race_names)
        self.cfg.n_race = len(self.race_names)
        self.heads_trained = self.cfg.attr_weight > 0
        self.model = SiameseCNN(self.cfg).to(self.device)
        self.model.load_state_dict(ck["model"]); self.model.eval()

    @staticmethod
    def _pil(image):
        if isinstance(image, (str, Path)):
            return Image.open(image).convert("RGB")
        if isinstance(image, Image.Image):
            return image.convert("RGB")
        arr = np.asarray(image)
        if arr.dtype != np.uint8:
            arr = (np.clip(arr, 0, 1) * 255).astype(np.uint8) if arr.max() <= 1.0 else arr.astype(np.uint8)
        if arr.ndim == 2:
            arr = np.stack([arr] * 3, -1)
        return Image.fromarray(arr[..., :3])

    @staticmethod
    def _largest_face(pil, pad=0.1):
        import cv2
        gray = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2GRAY)
        casc = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
        faces = casc.detectMultiScale(gray, 1.1, 5, minSize=(40, 40))
        if len(faces) == 0:
            return None, None
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        px, py = int(w * pad), int(h * pad)
        box = (max(x - px, 0), max(y - py, 0), min(x + w + px, pil.width), min(y + h + py, pil.height))
        return pil.crop(box), tuple(int(v) for v in box)

    def _tensor(self, image, detect_face):
        pil, info = self._pil(image), {"face_detected": None, "box": None}
        if detect_face:
            crop, box = self._largest_face(pil)
            info.update(face_detected=crop is not None, box=box)
            pil = crop if crop is not None else pil
        s = self.cfg.img_size
        arr = np.array(pil.resize((s, s), Image.BILINEAR), dtype=np.uint8)
        x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(self.device)
        return prep(x, False), info

    @torch.no_grad()
    def predict(self, image, detect_face=False):
        x, info = self._tensor(image, detect_face)
        ea, ei, g, r, a = self.model(x)
        pg, pr, pa = (F.softmax(t.float(), 1)[0].cpu().numpy() for t in (g, r, a))
        gi, ri, ai = int(pg.argmax()), int(pr.argmax()), int(pa.argmax())
        out = {**info, "gender": GENDER_NAMES[gi], "gender_confidence": float(pg[gi]),
               "race": self.race_names[ri], "race_confidence": float(pr[ri]),
               "race_probabilities": {n: float(p) for n, p in zip(self.race_names, pr)},
               "age_bin": self.age_names[ai], "age_bin_confidence": float(pa[ai]),
               "age_expected_years": float(pa @ AGE_BIN_MIDPOINTS),
               "embedding_attr": ea[0].cpu().numpy(), "embedding_inst": ei[0].cpu().numpy()}
        if not self.heads_trained:
            out["note"] = "attribute heads were not trained in this checkpoint (config B); use the probe readout"
        return out

    @torch.no_grad()
    def compare(self, image_a, image_b, detect_face=False):
        xa, ia = self._tensor(image_a, detect_face); xb, ib = self._tensor(image_b, detect_face)
        e = self.model(torch.cat([xa, xb]))
        d_attr = float((e[0][0] - e[0][1]).norm()); d_inst = float((e[1][0] - e[1][1]).norm())
        return {"attr_distance": d_attr, "attr_threshold": self.thr["attr"], "same_group": d_attr <= self.thr["attr"],
                "inst_distance": d_inst, "inst_threshold": self.thr["inst"], "same_image": d_inst <= self.thr["inst"],
                "face_a": ia, "face_b": ib}

_MODELS = {}
def get_face_model(checkpoint):
    key = str(Path(checkpoint).resolve())
    if key not in _MODELS:
        _MODELS[key] = SiameseFaceModel(checkpoint)
    return _MODELS[key]

def run_siamese_pipeline(image, checkpoint, reference=None, detect_face=False, include_embeddings=False):
    m = get_face_model(checkpoint)
    out = {"model": "siamese_cnn", "prediction": m.predict(image, detect_face), "comparison": None}
    if reference is not None:
        out["comparison"] = m.compare(image, reference, detect_face)
    if not include_embeddings:
        out["prediction"].pop("embedding_attr", None); out["prediction"].pop("embedding_inst", None)
    return out

**Validation - Step 13** (single-image path must agree with the batch evaluation path)

In [ ]:
def validate_inference_block(results: Dict[str, Any], data: Dict[str, pd.DataFrame], tracker: Optional[CheckTracker] = None):
    ckpt = results["checkpoint"]
    r_names = list(results["race"]["class_names"])
    te_ = data["test"].reset_index(drop=True)
    k = min(40, len(te_))
    out0 = run_siamese_pipeline(te_.path[0], ckpt, include_embeddings=True)
    p0 = out0["prediction"]
    check("prediction has the expected fields", {"gender", "race", "age_bin", "race_probabilities", "age_expected_years"} <= set(p0), tracker=tracker)
    check("race probabilities sum to 1", abs(sum(p0["race_probabilities"].values()) - 1) < 1e-4, tracker=tracker)
    check("embeddings are unit length", abs(np.linalg.norm(p0["embedding_attr"]) - 1) < 1e-3 and abs(np.linalg.norm(p0["embedding_inst"]) - 1) < 1e-3, tracker=tracker)
    agree = np.mean([run_siamese_pipeline(te_.path[i], ckpt)["prediction"]["gender"] == GENDER_NAMES[results["predictions"]["gender"][i]] for i in range(k)])
    check("single-image gender predictions agree with batch evaluation (>= 90%)", agree >= 0.9, f"{agree:.2f}", tracker=tracker)
    agree_r = np.mean([run_siamese_pipeline(te_.path[i], ckpt)["prediction"]["race"] == r_names[results["predictions"]["race"][i]] for i in range(k)])
    check("single-image race predictions agree with batch evaluation (>= 90%)", agree_r >= 0.9, f"{agree_r:.2f}", tracker=tracker)
    same = run_siamese_pipeline(te_.path[0], ckpt, reference=te_.path[0])["comparison"]
    check("an image compared with itself has ~0 distance and is 'same'", same["attr_distance"] < 1e-3 and same["same_group"] and same["same_image"], tracker=tracker)
    arr = np.asarray(Image.open(te_.path[0]).convert("RGB"))
    check("PIL / ndarray / path inputs give the same prediction",
          run_siamese_pipeline(Image.fromarray(arr), ckpt)["prediction"]["gender"] == run_siamese_pipeline(arr, ckpt)["prediction"]["gender"] == p0["gender"],
          tracker=tracker)

validate_inference_block(results, data)

## Step 14 - Unified end-to-end entrypoint (`run_experiment`)

`run_experiment(cfg=None, **overrides)` executes the **entire** pipeline end-to-end as a single function call:
1. Resolves dataset selection (`utkface`, `fairface`, `combined`), dataset sources (`utk_dir`, `fairface_source`, `fairface_config`), `race_schema` (`unified_5` or `native`), `epochs`, `steps_per_epoch`, learning rate, and runtime flags.
2. Runs dataset loading (UTKFace files, `HuggingFaceM4/FairFace` via Hugging Face `datasets`, or an uploaded FairFace `.zip`), canonical schema validation, `dhash` deduplication, near-duplicate audit, and stratified / official / cross-dataset splitting (`prepare_datasets`).
3. Builds/reuses disk-cached tensors, trains `C_full` (`run_siamese_cnn`), computes all classification reports, verification metrics, image-level bootstrap CIs, and fairness breakdowns.
4. Optionally runs multi-seed A/B/C ablation (`run_ablation=True`), renders diagnostic plots (`show_plots=True`), and validates every invariant (`run_validation=True`).

### Example Usage
```python
# 1. UTKFace run with custom epochs & steps_per_epoch
res = run_experiment(
    dataset="utkface",
    utk_dir="/kaggle/input/datasets/jangedoo/utkface-new/UTKFace",
    epochs=20,
    steps_per_epoch=300,
    show_plots=False,
)
display(res["metrics_table"])
display(res["classification_reports"]["race_df"])

# 2. FairFace run from HuggingFaceM4/FairFace (native 7-class race taxonomy)
res_ff = run_experiment(
    dataset="fairface",
    fairface_source="hf",          # or "/kaggle/input/.../fairface.zip" or "auto"
    fairface_config="0.25",        # "0.25" or "1.25"
    race_schema="native",
    split_mode="official",
    epochs=15,
    steps_per_epoch=400,
)

# 3. Cross-dataset generalization: train on FairFace, test on UTKFace
res_cross = run_experiment(
    dataset="fairface",
    eval_dataset="utkface",
    race_schema="unified_5",
    epochs=15,
)
```

In [ ]:
def run_experiment(cfg: Optional[Config] = None, verbose: bool = True, **overrides) -> Dict[str, Any]:
    """
    Run the complete end-to-end Siamese CNN pipeline from dataset ingestion to test evaluation.

    Parameters
    ----------
    cfg : Optional[Config]
        Base Config instance. If None, starts from default Config().
    verbose : bool
        Print progress, training logs, and final summary.
    **overrides :
        Any Config field (e.g. dataset, utk_dir, fairface_source, fairface_config,
        eval_dataset, race_schema, split_mode, epochs, steps_per_epoch, batch_pairs, batch_inst,
        lr, seed, quick_run, run_ablation, show_plots, run_validation, out_dir, cache_dir).

    Returns
    -------
    dict with keys:
        config, dataset_spec, data, metrics_table, classification_reports,
        verification, fairness, bootstrap, history, best_model_path,
        ablation, checks, summary, raw_results
    """
    run_cfg = make_config(base=cfg, **overrides)
    run_cfg.amp = run_cfg.amp and DEVICE.type == "cuda"
    Path(run_cfg.out_dir).mkdir(parents=True, exist_ok=True)
    if run_cfg.cache_dir:
        Path(run_cfg.cache_dir).mkdir(parents=True, exist_ok=True)

    tracker = CheckTracker(stop_on_fail=run_cfg.stop_on_fail, sync_global=False)

    # 1. Prepare datasets (load -> schema normalize -> dhash dedup -> split)
    prep = prepare_datasets(run_cfg, verbose=verbose)
    spec: DatasetSpec = prep["spec"]
    run_cfg.race_names = spec.race_names
    run_cfg.n_race = spec.num_races
    splits = {"train": prep["train"], "val": prep["val"], "test": prep["test"]}

    if run_cfg.run_validation:
        for sp_name, sp_df in splits.items():
            validate_schema(sp_df, f"{spec.name}[{sp_name}]", spec, tracker=tracker)

    # 2. Train & evaluate primary model
    model_cfg = dataclasses.replace(run_cfg, out_dir=str(Path(run_cfg.out_dir) / run_cfg.name))
    raw_res = run_siamese_cnn(splits, model_cfg, verbose=verbose)

    if run_cfg.run_validation:
        validate_results_block(raw_res, splits, run_cfg, tracker=tracker, check_config_b=False)
        validate_inference_block(raw_res, splits, tracker=tracker)

    # 3. Optional multi-seed A/B/C ablation
    ablation_out = None
    if run_cfg.run_ablation:
        ab_dir = Path(run_cfg.out_dir) / "ablation"
        ab_df = run_ablation(splits, run_cfg, ABLATIONS, ablation_dir=ab_dir, verbose=verbose)
        ablation_out = {
            "df": ab_df,
            "summary_table": format_ablation_table(ab_df),
            "csv_path": str(ab_dir / "ablation_results.csv"),
        }

    # 4. Optional diagnostic plots
    if run_cfg.show_plots:
        plot_diagnostics(raw_res)

    # 5. Assemble clean, analysis-ready return payload
    metrics_df = pd.DataFrame(raw_res["metrics_table"]).set_index("Task")
    cls_reports = {
        "gender": raw_res["gender"],
        "race": raw_res["race"],
        "age": raw_res["age"],
        "gender_df": per_class_metrics_df(raw_res, "gender"),
        "race_df": per_class_metrics_df(raw_res, "race"),
        "age_df": per_class_metrics_df(raw_res, "age"),
    }
    fairness_out = {
        "by_race_df": pd.DataFrame(raw_res["fairness"]["by_race"]).T,
        "by_gender_df": pd.DataFrame(raw_res["fairness"]["by_gender"]).T,
        "by_race_gender_df": pd.DataFrame(raw_res["fairness"]["by_race_gender"]).T,
        "summary": raw_res["fairness"]["summary"],
        "balanced": raw_res["fairness_balanced"],
    }
    history_df = pd.DataFrame(raw_res["history"])

    return {
        "config": asdict(run_cfg),
        "dataset_spec": {
            "name": spec.name,
            "race_schema": spec.race_schema,
            "race_names": list(spec.race_names),
            "n_race": spec.num_races,
            "n_age": spec.num_ages,
            "has_continuous_age": spec.has_continuous_age,
            "split_sizes": {k: len(v) for k, v in splits.items()},
            "duplicates_removed": prep["n_duplicates_removed"],
        },
        "data": splits,
        "metrics_table": metrics_df,
        "classification_reports": cls_reports,
        "verification": {
            "attribute": raw_res["verification"],
            "instance": raw_res["verification_instance"],
        },
        "fairness": fairness_out,
        "bootstrap": {
            "classification": raw_res["bootstrap"],
            "verification": raw_res["bootstrap_verification"],
        },
        "history": history_df,
        "best_model_path": raw_res["checkpoint"],
        "ablation": ablation_out,
        "checks": tracker.summary() if run_cfg.run_validation else None,
        "summary": raw_res["summary"],
        "raw_results": raw_res,
    }

# Alias for convenience
run_notebook_pipeline = run_experiment

## Check summary

All `PASS` / `FAIL` lines from the interactive session.

In [ ]:
S = pd.DataFrame(CHECKS)
if len(S):
    print(f"{int(S.passed.sum())} / {len(S)} checks passed")
    print(S[~S.passed][["check", "detail"]] if (~S.passed).any() else "no failures")
else:
    print("No interactive checks recorded yet.")